# Trajectory showcase: how much variety do N worlds give?

This notebook is a tool for one question: **with the parameter ranges below, how varied is a cohort of N generated worlds, and what is that variety made of?** It generates N worlds (10 by default; set `N_WORLDS = 100` to get every pair of channel and control counts once), saves them, and then reads only the saved file to plot, world by world:

1. how many channels and controls each world has — both from 1 to 10;
2. every parameter that generates the data, on the scale its prior is drawn on, against what that prior expects;
3. the trend parameter that makes an input flat, rising or falling, and the direction the series actually take;
4. how many weeks of each input are exactly zero;
5. how much of the outcome comes from channels, from controls and from the baseline.

Every reference is what the configured law predicts **at this N**, except for the outcome shares, whose reference is an ideal (an even spread of compositions) and is labelled so. With 10 worlds the bars move a lot by chance; at 100 worlds the same plots tighten. Change `N_WORLDS` or any range in the knobs cell and re-run to see how the distributions move.

**Units.** Model inputs are `treatment_raw` (channels, ≥ 0), `covariates` (signed controls) and `outcome_raw`. Parameters, schedules and contributions are privileged generator truth, never model features.

This revises the public example of [#29](https://github.com/pymc-labs/pymc-generator/issues/29) and the corpus report of [#31](https://github.com/pymc-labs/pymc-generator/issues/31): counts span 1–10 on both axes, every active input drives the outcome directly, and the report is plot-first.

## Install and run

Use **Python 3.13+** and the checkout containing this notebook (branched from PR #38). Older released packages can share the version number while lacking these features; the checkout revision and locked dependencies matter.

```bash
uv sync --locked --extra docs
.venv/bin/python -m ipykernel install --prefix "$PWD/.venv" --name pymc-generator
.venv/bin/python -m nbconvert --execute --to html \
  --ExecutePreprocessor.timeout=1800 --output-dir /tmp \
  docs/examples/dev/trajectory-showcase.ipynb
```

No GPU, posterior sampling or external data is needed. Set `PYMC_GENERATOR_SHOWCASE_DIR` to keep outputs in a directory you choose; otherwise they go to a new temporary directory.

In [ ]:
%matplotlib inline

import dataclasses
import hashlib
import itertools
import json
import math
import os
import tempfile
import warnings
from fractions import Fraction
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc
import pytensor
from matplotlib.ticker import PercentFormatter
from scipy import stats

import pymc_generator as pg
from pymc_generator.active_counts import allocation_targets
from pymc_generator.sampler import CARRYOVER_FAMILY_KEYS, SATURATION_FAMILY_KEYS, SCMPrior
from pymc_generator.signal_diagnostics import check_signal_gate
from pymc_generator.slots import (
    CORPUS_ARRAY_FIELDS,
    MECHANISM_ARRAY_FIELDS,
    MECHANISM_PRIOR_FIELDS,
    MECHANISM_REFERENCE_FIELDS,
    PRIOR_COND_LAYOUT,
    PRIOR_COND_QUANTITIES,
    TRAJECTORY_ARRAY_FIELDS,
    TRAJECTORY_COMPONENTS,
    TRAJECTORY_INPUTS,
    TRAJECTORY_PARAM_FIELDS,
    WALK_ARRAY_FIELDS,
    WALK_RELATIVE_MODE_FIELDS,
    SlotLayout,
)
from pymc_generator.trajectories import TEXTURE_COMPONENTS, flighting_on_weeks

GENERATION_SEED = 29
VISUAL_SEED = 31
VIF_THRESHOLD = 10.0
MAX_BAD_SHARE = 0.25
ENVELOPE_LEVEL = 0.9
DEFAULT_FLAT_BAND = 1 / 3

output_override = os.environ.get("PYMC_GENERATOR_SHOWCASE_DIR")
OUTPUT_DIR = (
    Path(output_override)
    if output_override
    else Path(tempfile.mkdtemp(prefix="pymc-generator-showcase-"))
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path(tempfile.mkdtemp(prefix="run-", dir=OUTPUT_DIR))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print(
    "Versions:",
    {"pymc-generator": pg.__version__, "pymc": pymc.__version__, "pytensor": pytensor.__version__},
)

## Report helpers

The next two cells define the report: loading a saved shard and its configuration file, the reference laws, and the figures. They read only saved files and generate nothing.

In [ ]:
# ---------------------------------------------------------------------------
# Report core: corpus + sidecar loading, masks, reference laws, the parameter
# grid and the count grid. Runs after the setup cell, which imports everything
# used here. Private names carry a `_core_` / `_CORE_` prefix so the figures
# helpers cannot shadow them.
# ---------------------------------------------------------------------------

SIDECAR_SUFFIX = ".config.json"

#: Written by ``write_sidecar``: the corpus came from this notebook's generation design.
_CORE_DESIGN_MARKER = "pymc-generator trajectory showcase: variety design v6"
_CORE_SIDECAR_KEYS = ("config", "design", "knobs", "sha256")
_CORE_NOUN = {"treatment": "channel", "covariate": "control"}
_CORE_COLORS = {"treatment": "#306998", "covariate": "#d0782a", "world": "#5b8c5a"}
_CORE_EDGE_LABELS = {
    "cy": "C→Y",
    "dc": "D→C",
    "dz": "D→Z",
    "dy": "D→Y",
    "zy": "Z→Y",
    "zc": "Z→C",
    "cc": "C→C",
    "zz": "Z→Z",
}
#: Panel titles of the response-shape draws (keys of MECHANISM_PRIOR_FIELDS).
_CORE_SHAPE_TITLES = {
    "hill_slope": "Hill slope",
    "hill_kappa_mult": "Hill κ multiplier",
    "logistic_lam": "Logistic λ",
    "mm_kappa_mult": "Michaelis–Menten κ multiplier",
    "tanh_c": "Tanh c",
    "root_alpha": "Root α",
}
#: Universe (1), metadata class: top-level corpus keys outside the array registries.
_CORE_METADATA_KEYS = frozenset({"prior_cond", "identifiability", "diagnostics"})
#: Universe (2): every key the library writes into ``corpus["diagnostics"]``.
_CORE_DIAGNOSTICS_KEYS = frozenset(
    {
        "active_count_coverage",
        "baseline_decomposition_max_abs_error",
        "decomposition_max_abs_error",
        "draws_per_cell",
        "edge_base_rates",
        "edge_budget",
        "edge_marginals",
        "edge_types",
        "full_decomposition_max_abs_error",
        "mechanism_priors",
        "min_no_direct_effect_treatments",
        "n_cells",
        "n_draw_failures",
        "n_draws_evaluated",
        "n_tasks",
        "prior_cond",
        "rejection_rate",
        "schema_version",
        "short_horizon_n_query",
        "signal",
        "telescoping_split_max_abs_error",
        "timing",
        "trajectory",
        "treatment_cv_quantiles",
        "treatment_share_quantiles",
        "walk_parameters",
    }
)
#: CORPUS_ARRAY_FIELDS split into generation parameters (plotted in the grid) and
#: realized outputs / bookkeeping. Asserted to partition the registry exactly, so a
#: new library field stops the report instead of silently missing from the grid.
_CORE_PARAMETER_KEYS = frozenset(
    {
        "g",
        "n_treatments_active",
        "n_covariates_active",
        "n_latent_active",
        "carryover_family",
        "carryover_alpha",
        "weibull_lam",
        "weibull_k",
        "treatment_level",
        "saturation_scale",
        "confounding_strength",
        "treatment_shock_index",
        "treatment_shock_start",
        "treatment_shock_length",
        "treatment_shock_level_multiplier",
        "treatment_shock_level",
    }
)
_CORE_OUTPUT_KEYS = frozenset(
    {
        "treatment_raw",
        "treatment_norm",
        "treatment_share",
        "covariates",
        "outcome_raw",
        "outcome_norm",
        "support_mask",
        "is_future",
        "treatment_contribution_raw",
        "baseline_raw",
        "latent_unobserved",
        "treatment_means",
        "outcome_scale",
        "is_val",
        "cell_id",
        "treatment_active_mask",
        "covariate_active_mask",
        "latent_active_mask",
        "indirect_effects",
        "treatment_active",
        "covariate_contribution",
        "latent_unobserved_contribution",
        "baseline_intrinsic",
        "outcome_noise",
        "indirect_effects_by_source",
        "treatment_shock_mask",
    }
)
#: Support-check tolerance per value source: (machine epsilon, ulps).
_CORE_PRECISION = {
    "float32": (float(np.finfo(np.float32).eps), 4.0),
    "float64": (float(np.finfo(np.float64).eps), 4.0),
    "ratio": (float(np.finfo(np.float64).eps), 16.0),
}
_CORE_GROUPS = {
    "structure": "Structure drawn once per cell",
    "walk": "Walk and noise scales",
    "mechanism": "Response mechanisms and effect sizes",
    "treatment": "Channel trajectory components (pooled over carriers)",
    "covariate": "Control trajectory components (pooled over carriers)",
    "events": "Shocks and conditioning intervals",
}
_CORE_DRAWABLE = ("reference", "derived", "unavailable", "bars")


class SidecarMismatchError(RuntimeError):
    """The corpus file's SHA-256 differs from the hash its sidecar records."""


class SidecarConfigMismatchError(RuntimeError):
    """The hash matches, but the sidecar config disagrees with the corpus diagnostics."""


@dataclasses.dataclass(frozen=True)
class Report:
    """One validated corpus and what the report knows about how it was generated."""

    path: Path = dataclasses.field(repr=False)
    corpus: dict = dataclasses.field(repr=False)
    config: dict | None = dataclasses.field(repr=False)
    knobs: dict | None
    design: bool
    walk: bool
    T: int
    n_worlds: int
    acceptance: dict
    notes: list[str]


def _core_require(condition, message):
    """Raise AssertionError(message) unless condition holds (never stripped by -O)."""
    if not condition:
        raise AssertionError(message)


def _core_check_role(role):
    if role not in TRAJECTORY_INPUTS:
        raise ValueError(f"role must be one of {TRAJECTORY_INPUTS}, got {role!r}")


# --- sidecar ----------------------------------------------------------------------


def _core_sidecar_path(corpus_path):
    """``<name>.config.json`` next to ``<name>.npz``."""
    return Path(corpus_path).with_suffix(SIDECAR_SUFFIX)


def _core_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _core_json_normalize(value, where):
    """``value`` as plain JSON types (tuples become lists), naming any offending field."""
    if isinstance(value, dict):
        out = {}
        for key, item in value.items():
            if not isinstance(key, str):
                raise TypeError(f"{where} key {key!r} must be a string")
            out[key] = _core_json_normalize(item, f"{where}.{key}")
        return out
    if isinstance(value, (list, tuple)):
        return [_core_json_normalize(item, f"{where}[{i}]") for i, item in enumerate(value)]
    if isinstance(value, np.ndarray):
        return _core_json_normalize(value.tolist(), where)
    if isinstance(value, np.generic):
        value = value.item()
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if not math.isfinite(value):
            raise ValueError(f"{where} must be finite to be stored as JSON, got {value!r}")
        return value
    raise TypeError(f"{where} has type {type(value).__name__}, which JSON cannot store")


def write_sidecar(corpus_path: Path, cfg: SCMPrior, knobs: dict) -> Path:
    """Write ``<name>.config.json`` next to a saved corpus and return its path.

    The sidecar holds the JSON-normalized ``dataclasses.asdict(cfg)`` (tuples become
    lists, so ``SCMPrior(**config)`` rebuilds the config), the notebook knobs, this
    notebook's design marker and the SHA-256 of the ``.npz`` bytes.
    """
    corpus_path = Path(corpus_path)
    if not isinstance(cfg, SCMPrior):
        raise TypeError(f"cfg must be an SCMPrior, got {type(cfg).__name__}")
    if not isinstance(knobs, dict):
        raise TypeError(f"knobs must be a dict, got {type(knobs).__name__}")
    if corpus_path.suffix != ".npz" or not corpus_path.is_file():
        raise FileNotFoundError(
            f"corpus_path must name a saved .npz corpus, got {corpus_path.name}"
        )
    config = _core_json_normalize(dataclasses.asdict(cfg), "config")
    if _core_json_normalize(dataclasses.asdict(SCMPrior(**config)), "config") != config:
        raise ValueError("config does not survive the round trip SCMPrior(**config)")
    payload = {
        "config": config,
        "design": _CORE_DESIGN_MARKER,
        "knobs": _core_json_normalize(knobs, "knobs"),
        "sha256": _core_sha256(corpus_path),
    }
    path = _core_sidecar_path(corpus_path)
    # Keep insertion order: SCMPrior(**config) then rebuilds every nested dict in the
    # generator's own order, so regenerating from the sidecar writes identical bytes.
    text = json.dumps(payload, indent=2, allow_nan=False)
    path.write_text(text + "\n", encoding="utf-8")
    return path


def _core_read_sidecar(path):
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"sidecar {path.name} is not valid JSON") from exc
    if not isinstance(payload, dict) or tuple(sorted(payload)) != _CORE_SIDECAR_KEYS:
        raise ValueError(
            f"sidecar {path.name} must hold exactly the keys {list(_CORE_SIDECAR_KEYS)}"
        )
    digest = payload["sha256"]
    if not isinstance(digest, str) or len(digest) != 64 or set(digest) - set("0123456789abcdef"):
        raise ValueError(f"sidecar {path.name} sha256 must be a 64-digit lowercase hex digest")
    for key in ("config", "knobs"):
        if not isinstance(payload[key], dict):
            raise ValueError(f"sidecar {path.name} {key} must be a JSON object")
    if not isinstance(payload["design"], str):
        raise ValueError(f"sidecar {path.name} design must be a string")
    return payload


def _core_mechanism_echo(cfg):
    """``diagnostics["mechanism_priors"]`` as the library writes it for ``cfg``."""
    fields = list(MECHANISM_ARRAY_FIELDS)
    for role in TRAJECTORY_INPUTS:
        if getattr(cfg, f"{role}_reference_contribution_range") is not None:
            fields += [key for key in MECHANISM_REFERENCE_FIELDS if key.startswith(f"{role}_")]

    def bounds(value):
        return None if value is None else [float(v) for v in value]

    return {
        "parameter_fields": fields,
        "saturation_prior_ranges": {
            family: {name: [float(lo), float(hi)] for name, (lo, hi) in parameters.items()}
            for family, parameters in cfg.saturation_prior_ranges.items()
        },
        "mm_scale_prior": cfg.mm_scale_prior,
        "treatment_reference_contribution_range": bounds(
            cfg.treatment_reference_contribution_range
        ),
        "treatment_reference_multiplier": float(cfg.treatment_reference_multiplier),
        "covariate_reference_contribution_range": bounds(
            cfg.covariate_reference_contribution_range
        ),
        "covariate_reference_scale": float(cfg.covariate_reference_scale),
    }


def _core_prior_cond_echo(cfg):
    """``diagnostics["prior_cond"]`` as the library writes it for ``cfg``."""
    spec = cfg.prior_cond_spec()
    return {
        "layout": list(PRIOR_COND_LAYOUT),
        "supports": {q: list(s["support"]) for q, s in spec.items()},
        "width_ranges": {q: list(s["width_range"]) for q, s in spec.items()},
    }


def _core_check_sidecar_config(corpus, config, *, design, name):
    """Rebuild ``SCMPrior(**config)``, compare every diagnostics echo, return the prior."""
    names = {field.name for field in dataclasses.fields(SCMPrior)}
    unknown = sorted(set(config) - names)
    if unknown:
        raise SidecarConfigMismatchError(
            f"{name}: sidecar config fields SCMPrior does not define: {unknown}"
        )
    if design:
        missing = sorted(names - set(config))
        if missing:
            raise SidecarConfigMismatchError(
                f"{name}: sidecar config lacks SCMPrior fields: {missing}"
            )
    try:
        cfg = SCMPrior(**config)
        cfg.validate()
    except (TypeError, ValueError) as exc:
        raise SidecarConfigMismatchError(
            f"{name}: sidecar config does not rebuild a valid SCMPrior: {exc}"
        ) from exc
    diagnostics = corpus["diagnostics"]
    budget = (
        {
            edge: list(value) if isinstance(value, (tuple, list)) else int(value)
            for edge, value in cfg.edge_budget.items()
        }
        if cfg.edge_budget
        else None
    )
    mismatched = [
        label
        for label, stored, rebuilt in (
            ("n_time_steps", corpus["outcome_raw"].shape[1], int(cfg.n_time_steps)),
            ("n_treatments", corpus["treatment_active_mask"].shape[1], int(cfg.n_treatments)),
            ("n_covariates", corpus["covariate_active_mask"].shape[1], int(cfg.n_covariates)),
            ("n_latent", corpus["latent_active_mask"].shape[1], int(cfg.n_latent)),
            ("draws_per_cell", diagnostics.get("draws_per_cell"), int(cfg.draws_per_cell)),
            ("n_cells", diagnostics.get("n_cells"), int(cfg.n_cells)),
            ("edge_budget", diagnostics.get("edge_budget"), budget),
        )
        if stored != rebuilt
    ]
    for label, enabled, rebuild in (
        ("mechanism_priors", cfg.mechanism_priors_enabled, lambda: _core_mechanism_echo(cfg)),
        ("walk_parameters", cfg.record_walk_parameters, cfg.walk_parameter_metadata),
        ("prior_cond", cfg.prior_conditioning, lambda: _core_prior_cond_echo(cfg)),
    ):
        if (label in diagnostics) != bool(enabled) or (enabled and diagnostics[label] != rebuild()):
            mismatched.append(label)
    trajectory = diagnostics.get("trajectory")
    if (trajectory is not None) != cfg.trajectory_metadata_enabled:
        mismatched.append("trajectory")
    elif trajectory is not None:
        if trajectory.get("inclusion_probs") != cfg.trajectory_inclusion_probs():
            mismatched.append("trajectory.inclusion_probs")
        jump_counts = {
            role: int(getattr(cfg, f"{role}_level_jump_count")) for role in TRAJECTORY_INPUTS
        }
        if trajectory.get("jump_counts") != jump_counts:
            mismatched.append("trajectory.jump_counts")
    coverage = diagnostics.get("active_count_coverage")
    stratified = cfg.active_count_allocation == "stratified"
    if (coverage is not None) != stratified:
        mismatched.append("active_count_coverage")
    elif stratified and coverage.get("weights") != cfg.active_count_weight_matrix().tolist():
        mismatched.append("active_count_coverage.weights")
    if mismatched:
        raise SidecarConfigMismatchError(
            f"{name}: the sidecar config disagrees with the corpus on {', '.join(mismatched)}"
        )
    return cfg


def _core_prior(report):
    """The rebuilt ``SCMPrior(**config)`` the configured laws come from, or None."""
    return None if report.config is None else SCMPrior(**report.config)


def _core_zero_law_blocker(cfg, role):
    """Why the exact zero-week law is unavailable for ``role``, or None."""
    if cfg is None:
        return "configured law unavailable: no sidecar config"
    probs = cfg.trajectory_inclusion_probs()[role]
    if (
        probs["onset"] != 0.0
        or probs["offset"] != 0.0
        or (role == "treatment" and int(cfg.n_treatment_shocks) != 0)
    ):
        return "configured law unavailable: gates other than flighting"
    return None


def _core_notes(corpus, cfg, *, sidecar, design):
    diagnostics = corpus["diagnostics"]
    notes = []
    if not sidecar:
        notes.append(
            "configured law unavailable: no sidecar config, so parameter, trend, zero-week "
            "and flag panels show histograms only"
        )
        notes.append("design assertions skipped: no sidecar marks this notebook's design")
    elif not design:
        notes.append("design assertions skipped: the sidecar does not mark this notebook's design")
    if "trajectory" not in diagnostics:
        notes.append(
            "no trajectory block (no schedule component could be included): component "
            "prevalence, carrier pooling, the zero identity and trend parameters are unavailable"
        )
    if "walk_parameters" not in diagnostics:
        notes.append(
            "walk and noise parameters not recorded (generated without record_walk_parameters=True)"
        )
    if "mechanism_priors" not in diagnostics:
        notes.append("mechanism block not recorded: response-shape parameters are not recorded")
    if "active_count_coverage" not in diagnostics:
        notes.append("no stored active-count coverage block to compare the count grid with")
    draws_per_cell = int(diagnostics["draws_per_cell"])
    if draws_per_cell > 1:
        notes.append(
            f"draws_per_cell = {draws_per_cell}: cell-level panels use one row per cell and "
            "carry no envelope"
        )
    if cfg is not None:
        for role in TRAJECTORY_INPUTS:
            blocker = _core_zero_law_blocker(cfg, role)
            if blocker is not None:
                notes.append(f"zero-week reference for {_CORE_NOUN[role]}s: {blocker}")
    return notes


def load_report(path: Path) -> Report:
    """Load and validate a corpus, cross-check its sidecar, and degrade per the plan.

    Raises ``ValueError`` when ``validate_corpus`` reports errors,
    ``SidecarMismatchError`` when the sidecar's SHA-256 does not match the file, and
    ``SidecarConfigMismatchError`` when the hash matches but a diagnostics echo differs
    from what the rebuilt ``SCMPrior(**config)`` produces. Without a sidecar every
    configured-law reference degrades to "configured law unavailable". A sidecar that
    marks this notebook's design also runs the design assertions.
    """
    path = Path(path)
    corpus = pg.load_corpus(path)
    errors = pg.DataGenerator.validate_corpus(corpus)
    if errors:
        raise ValueError(f"{path.name} fails validate_corpus: " + "; ".join(errors))
    config = knobs = cfg = None
    design = False
    sidecar = _core_sidecar_path(path)
    if sidecar.exists():
        payload = _core_read_sidecar(sidecar)
        digest = _core_sha256(path)
        if digest != payload["sha256"]:
            raise SidecarMismatchError(
                f"SHA-256 of {path.name} is {digest}, "
                f"but {sidecar.name} records {payload['sha256']}"
            )
        design = payload["design"] == _CORE_DESIGN_MARKER
        config, knobs = payload["config"], payload["knobs"]
        cfg = _core_check_sidecar_config(corpus, config, design=design, name=sidecar.name)
    diagnostics = corpus["diagnostics"]
    report = Report(
        path=path,
        corpus=corpus,
        config=config,
        knobs=knobs,
        design=design,
        walk="walk_parameters" in diagnostics,
        T=int(corpus["outcome_raw"].shape[1]),
        n_worlds=int(corpus["outcome_raw"].shape[0]),
        acceptance={
            "evaluated": diagnostics.get("n_draws_evaluated"),
            "rejection_rate": diagnostics.get("rejection_rate"),
        },
        notes=_core_notes(corpus, cfg, sidecar=config is not None, design=design),
    )
    if design:
        _core_design_assertions(report)
    return report


def _core_integer_matrix(value):
    """``value`` as int64 if it is a rectangular nested list of ints >= 0, else None."""
    if not isinstance(value, list) or not value or not all(isinstance(row, list) for row in value):
        return None
    if len({len(row) for row in value}) != 1 or not value[0]:
        return None
    flat = [item for row in value for item in row]
    if not all(isinstance(item, int) and not isinstance(item, bool) and item >= 0 for item in flat):
        return None
    return np.asarray(value, dtype=np.int64)


def _core_design_weights(report):
    """The design count matrix W (int64) when a sidecar marks this design, else None."""
    if not report.design:
        return None
    weights = _core_integer_matrix(report.knobs.get("count_weights"))
    _core_require(weights is not None, "design knobs count_weights must be an integer matrix")
    return weights.astype(np.int64)


def _core_design_assertions(report):
    """The checks a sidecar marking this notebook's design enables (plan: design assertions)."""
    cfg = _core_prior(report)
    corpus, knobs = report.corpus, report.knobs
    diagnostics = corpus["diagnostics"]
    weights = _core_design_weights(report)
    treatment_axis, covariate_axis = cfg.active_count_grid
    _core_require(
        weights.shape == (len(treatment_axis), len(covariate_axis)),
        f"count_weights shape {weights.shape} must match the active-count grid "
        f"{len(treatment_axis)}x{len(covariate_axis)}",
    )
    _core_require(
        int(weights.sum()) == int(cfg.n_cells),
        f"count_weights must sum to n_cells={cfg.n_cells}, got {int(weights.sum())}",
    )
    _core_require(
        len(set(weights.sum(axis=1).tolist())) == 1 and len(set(weights.sum(axis=0).tolist())) == 1,
        "count_weights must have equal row sums and equal column sums",
    )
    _core_require(
        cfg.active_count_allocation == "stratified"
        and report.config["active_count_weights"] == knobs["count_weights"],
        "the design allocates cells by active_count_weights == count_weights (stratified)",
    )
    coverage = diagnostics.get("active_count_coverage")
    _core_require(
        coverage is not None
        and np.array_equal(np.asarray(coverage["n_cells"], dtype=np.int64), weights),
        "realized cells per count combination must equal the design count_weights",
    )
    _core_require(
        int(cfg.draws_per_cell) == 1 and int(diagnostics["draws_per_cell"]) == 1,
        "the design fixes draws_per_cell = 1",
    )
    _core_require(
        cfg.outcome_std_mode == "relative", "the design fixes outcome_std_mode='relative'"
    )
    _core_require(
        report.n_worlds == int(cfg.n_cells) == knobs.get("N_WORLDS"),
        "the design generates exactly N_WORLDS = n_cells worlds (never n=), got "
        f"{report.n_worlds} worlds, n_cells={cfg.n_cells}, N_WORLDS={knobs.get('N_WORLDS')!r}",
    )
    layout = cfg.layout
    g = np.asarray(corpus["g"])
    _core_require(
        np.array_equal(g[:, layout.slices["cy"]], corpus["treatment_active_mask"]),
        "every active channel must have its C→Y edge, and only active channels",
    )
    _core_require(
        np.array_equal(g[:, layout.slices["zy"]], corpus["covariate_active_mask"]),
        "every active control must have its Z→Y edge, and only active controls",
    )
    others = [edge for edge in layout.edge_types if edge not in ("cy", "zy")]
    _core_require(
        not any(g[:, layout.slices[edge]].any() for edge in others),
        f"edge types {others} must be empty in the design (latent disconnected)",
    )
    for key in ("latent_unobserved_contribution", "indirect_effects", "indirect_effects_by_source"):
        _core_require(
            not np.asarray(corpus[key]).any(), f"{key} must be identically 0 in the design"
        )
    _core_require(int(cfg.n_treatment_shocks) == 0, "the design has no treatment shocks")
    probs = cfg.trajectory_inclusion_probs()
    for role in TRAJECTORY_INPUTS:
        for gate in ("onset", "offset"):
            carried = carried_mask(report, role, gate)
            # Without a trajectory block (every schedule probability 0) no input carries a gate.
            _core_require(
                probs[role][gate] == 0.0 and (carried is None or not carried.any()),
                f"the design sets {role}_{gate}_inclusion_prob = 0 and no input carries {gate}",
            )
        _core_require(
            exact_zero_law(report, role) is not None,
            f"the {role} zero-week reference must be the flighting-only law",
        )


# --- masks and reference envelopes ------------------------------------------------


def active_mask(report, role) -> np.ndarray:
    """(worlds, slots) bool: which ``role`` slots are active."""
    _core_check_role(role)
    return np.asarray(report.corpus[f"{role}_active_mask"]) == 1


def carried_mask(report, role, component) -> np.ndarray | None:
    """Active slots that carry ``component``, or None without a trajectory block."""
    _core_check_role(role)
    if component not in TRAJECTORY_COMPONENTS:
        raise ValueError(f"component must be one of {TRAJECTORY_COMPONENTS}, got {component!r}")
    key = f"{role}_components"
    if key not in report.corpus:
        return None
    flags = np.asarray(report.corpus[key])[..., TRAJECTORY_COMPONENTS.index(component)] == 1
    return active_mask(report, role) & flags


def sidak_envelope(probs, n, level=ENVELOPE_LEVEL) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Expected counts and a Šidák family-wise envelope from exact binomial quantiles.

    Each entry is its own Binomial(n, p_i) marginal (a multinomial bin or an
    independent test); with k entries the per-entry two-sided error is
    ``1 - level ** (1 / k)``, so all k counts fall inside together with probability
    at least ``level``. Returns (expected, lo, hi).
    """
    probs = np.asarray(probs, dtype=np.float64)
    if probs.ndim != 1 or probs.size == 0:
        raise ValueError(f"probs must be a non-empty 1-D array, got shape {probs.shape}")
    if not np.all((probs >= 0.0) & (probs <= 1.0)):
        raise ValueError(f"probs must lie in [0, 1], got {probs.tolist()}")
    if (
        isinstance(n, (bool, np.bool_))
        or not isinstance(n, (int, float, np.integer, np.floating))
        or n < 0
        or n != int(n)
    ):
        raise ValueError(f"n must be a non-negative integer, got {n!r}")
    n = int(n)
    if not 0.0 < level < 1.0:
        raise ValueError(f"level must lie in (0, 1), got {level!r}")
    alpha = 1.0 - level ** (1.0 / probs.size)
    expected = n * probs
    lo = stats.binom.ppf(alpha / 2.0, n, probs)
    hi = stats.binom.isf(alpha / 2.0, n, probs)
    return expected, lo, hi


def _core_unit(probs):
    """Category masses as float64 summing to 1, within the library's 1e-8 tolerance."""
    probs = np.asarray(probs, dtype=np.float64)
    total = float(probs.sum())
    _core_require(abs(total - 1.0) <= 1e-8, "a panel's category masses must sum to 1")
    return probs / total


def _core_family_rate(probs, n, *, bernoulli, slots=None, draws=20_000):
    """Chance that a panel drawn from its own law leaves its Šidák envelope somewhere.

    Independent Bernoulli categories (component flags) are exact. Otherwise a seeded
    simulation (``VISUAL_SEED``) of ``draws`` panels: one multinomial over the pooled
    values, or, for one event per slot (``slots``: level-jump weeks, shock starts), the
    sum of one multinomial per slot. At most ``1 - ENVELOPE_LEVEL`` by construction, and
    smaller wherever the exact binomial quantiles or the slot structure are conservative.
    """
    probs = np.asarray(probs, dtype=np.float64)
    n = int(n)
    _, lo, hi = sidak_envelope(probs, n)
    if bernoulli:
        inside = stats.binom.cdf(hi, n, probs) - stats.binom.cdf(lo - 1, n, probs)
        return float(1.0 - np.prod(inside))
    rng = np.random.default_rng(VISUAL_SEED)
    if slots:
        per_slot, remainder = divmod(n, len(slots))
        _core_require(remainder == 0, "a slotted panel pools one value per slot and input")
        counts = sum(rng.multinomial(per_slot, _core_unit(p), size=draws) for p in slots)
    else:
        counts = rng.multinomial(n, _core_unit(probs), size=draws)
    return float(((counts < lo) | (counts > hi)).any(axis=1).mean())


def _core_envelope_marks(ax, x, expected, lo, hi):
    ax.vlines(x, lo, hi, color="black", linewidth=1.0, zorder=3)
    ax.scatter(x, lo, marker="_", s=40, color="black", zorder=3, linewidths=1.0)
    ax.scatter(x, hi, marker="_", s=40, color="black", zorder=3, linewidths=1.0)
    ax.scatter(x, expected, s=12, color="black", zorder=4)


def _core_category_ticks(ax, x, labels):
    rotate = len(x) > 6 or max(len(str(label)) for label in labels) > 6
    ax.set_xticks(
        x,
        [str(label) for label in labels],
        rotation=40 if rotate else 0,
        ha="right" if rotate else "center",
    )
    ax.tick_params(labelsize=7)


def draw_reference_bars(ax, counts, probs, n, labels, *, color, title):
    """Categorical bars, expected counts (dots) and the Šidák envelope (whiskers).

    Bars outside their envelope get a red edge; the title gains "n …" and
    "k of m outside". Returns ``(k, m)``.
    """
    counts = np.asarray(counts)
    probs = np.asarray(probs, dtype=np.float64)
    if counts.ndim != 1 or counts.shape != probs.shape:
        raise ValueError(
            f"counts and probs must be 1-D with equal length, got {counts.shape} and {probs.shape}"
        )
    if len(labels) != counts.size:
        raise ValueError(f"labels must name all {counts.size} bars, got {len(labels)}")
    expected, lo, hi = sidak_envelope(probs, n)
    outside = (counts < lo) | (counts > hi)
    x = np.arange(counts.size)
    ax.bar(
        x,
        counts,
        width=0.75,
        color=color,
        edgecolor=["#c0392b" if out else color for out in outside],
        linewidth=1.5,
        zorder=2,
    )
    _core_envelope_marks(ax, x, expected, lo, hi)
    _core_category_ticks(ax, x, labels)
    k = int(outside.sum())
    ax.set_title(f"{title} · n {int(n)}\n{k} of {counts.size} outside", fontsize=8)
    return k, int(counts.size)


# --- zero weeks and trend categories ----------------------------------------------


def zero_category_edges(T) -> tuple[np.ndarray, list[str]]:
    """Zero-week categories for a T-week window: [0, 1] + range(11, T+1, 10) + [T+1].

    Category i holds counts in [edges[i], edges[i+1]); labels read "0", "1–10", …
    """
    if isinstance(T, (bool, np.bool_)) or not isinstance(T, (int, np.integer)) or T < 1:
        raise ValueError(f"T must be a positive integer, got {T!r}")
    T = int(T)
    edges = np.asarray([0, 1, *range(11, T + 1, 10), T + 1], dtype=np.int64)
    labels = [
        str(lo) if hi - lo == 1 else f"{lo}–{hi - 1}"
        for lo, hi in zip(edges[:-1].tolist(), edges[1:].tolist())
    ]
    return edges, labels


def _core_bin_index(values, edges):
    """Left-closed bin of each value; the top edge and in-tolerance overhang join end bins."""
    index = np.searchsorted(edges, values, side="right") - 1
    return np.clip(index, 0, len(edges) - 2)


def _core_on_weeks_law(period, duty_lo, duty_hi):
    """Exact law {W: mass} of the on-run length given period P, duty ~ U(duty_lo, duty_hi).

    ``flighting_on_weeks(duty, P)`` is a step function of duty that changes only where
    ``duty·P + 0.5`` crosses an integer, i.e. at duty = (2k − 1) / (2P). Each
    sub-interval between consecutive cut points carries its share of the duty range
    and its W is read from the library function at the sub-interval midpoint. A fixed
    duty is a point mass.
    """
    if duty_lo == duty_hi:
        return {int(flighting_on_weeks(duty_lo, period)): Fraction(1)}
    lo, hi = Fraction(duty_lo), Fraction(duty_hi)
    points = {lo, hi}
    for k in range(math.floor(duty_lo * period), math.floor(duty_hi * period + 0.5) + 2):
        cut = Fraction(2 * k - 1, 2 * period)
        if lo < cut < hi:
            points.add(cut)
    ordered = sorted(points)
    law = {}
    for a, b in zip(ordered[:-1], ordered[1:]):
        on_weeks = int(flighting_on_weeks(float((a + b) / 2), period))
        law[on_weeks] = law.get(on_weeks, Fraction(0)) + (b - a) / (hi - lo)
    return law


def _core_flighting_law(cfg, role):
    """Exact joint law {(P, W): mass} of a carrier's flighting period and on-run length."""
    p_lo, p_hi = (int(v) for v in getattr(cfg, f"{role}_flighting_period_weeks_range"))
    d_lo, d_hi = (float(v) for v in getattr(cfg, f"{role}_flighting_duty_range"))
    share = Fraction(1, p_hi - p_lo + 1)
    return {
        (period, on_weeks): share * mass
        for period in range(p_lo, p_hi + 1)
        for on_weeks, mass in _core_on_weeks_law(period, d_lo, d_hi).items()
    }


def _core_flighting_zero_law(cfg, role, T):
    """Exact law {zero weeks: mass} of a flighting carrier over reported weeks 0..T-1.

    Week t is on iff (t + phase) mod P < W, with phase uniform on 0..P-1.
    """
    weeks = np.arange(T)
    law = {}
    for (period, on_weeks), mass in _core_flighting_law(cfg, role).items():
        phases = np.arange(period)
        on = (((weeks[None, :] + phases[:, None]) % period) < on_weeks).sum(axis=1)
        zeros, counts = np.unique(T - on, return_counts=True)
        for zero, count in zip(zeros.tolist(), counts.tolist()):
            law[zero] = law.get(zero, Fraction(0)) + mass * Fraction(count, period)
    return law


def exact_zero_law(report, role) -> np.ndarray | None:
    """Exact category masses of zero weeks per active series (incl. "0"), or None.

    Requires the sidecar config with onset = offset = 0 for ``role`` (treatments
    also n_treatment_shocks == 0): a series is then zero exactly on its flighting
    off-weeks, so the law is the mixture of "0" (no flighting, 1 − inclusion) with
    the period × duty × phase push-forward of a flighting carrier. Masses sum to 1
    within 1e-12 (exactly in rational arithmetic).
    """
    _core_check_role(role)
    cfg = _core_prior(report)
    if _core_zero_law_blocker(cfg, role) is not None:
        return None
    edges, _ = zero_category_edges(report.T)
    inclusion = Fraction(cfg.trajectory_inclusion_probs()[role]["flighting"])
    masses = [Fraction(0)] * (len(edges) - 1)
    masses[0] += 1 - inclusion
    if inclusion > 0:
        for zero, mass in _core_flighting_zero_law(cfg, role, report.T).items():
            masses[int(_core_bin_index(np.asarray([zero]), edges)[0])] += inclusion * mass
    _core_require(sum(masses) == 1, f"{role} zero-week law masses must sum to 1 exactly")
    out = np.asarray([float(mass) for mass in masses])
    _core_require(abs(out.sum() - 1.0) <= 1e-12, f"{role} zero-week law must sum to 1 within 1e-12")
    return out


def trend_category_law(lo, hi, band) -> np.ndarray:
    """Masses (down, flat, up) of U(lo, hi): down x < −band, flat |x| ≤ band, up x > band.

    Conditional on carrying the trend; lo == hi is a point mass.
    """
    lo, hi, band = float(lo), float(hi), float(band)
    if not (math.isfinite(lo) and math.isfinite(hi) and lo <= hi):
        raise ValueError(f"trend range must be finite with lo <= hi, got ({lo}, {hi})")
    if not (math.isfinite(band) and band >= 0.0):
        raise ValueError(f"band must be finite and >= 0, got {band}")
    if lo == hi:
        return np.asarray([lo < -band, -band <= lo <= band, lo > band], dtype=np.float64)
    width = hi - lo
    down = max(0.0, min(hi, -band) - lo) / width
    up = max(0.0, hi - max(lo, band)) / width
    flat = max(0.0, min(hi, band) - max(lo, -band)) / width
    return np.asarray([down, flat, up], dtype=np.float64)


# --- shares -----------------------------------------------------------------------


def realized_shares(report, *, show_closure=False) -> dict[str, np.ndarray]:
    """Per-world channel / control / baseline shares of total outcome (float64).

    channel = Σ treatment_contribution_raw + Σ indirect_effects; control =
    Σ covariate_contribution; baseline = Σ baseline_intrinsic + Σ latent contribution
    + Σ outcome_noise; each divided by Σ outcome_raw (> 0). Asserts the per-world
    closure within the float32 storage bound, the regrouping against the library's
    ``additive_share_total()`` and the channel share against the library's
    ``treatment_total_contribution`` unit_share. ``show_closure`` prints the largest
    closure residual.
    """
    corpus = report.corpus

    def load(key):
        return np.asarray(corpus[key], dtype=np.float64)

    outcome = load("outcome_raw")
    treatment, indirect = load("treatment_contribution_raw"), load("indirect_effects")
    control, intrinsic = load("covariate_contribution"), load("baseline_intrinsic")
    latent, noise = load("latent_unobserved_contribution"), load("outcome_noise")
    sources = load("indirect_effects_by_source")
    outcome_total = outcome.sum(axis=1)
    bad = np.flatnonzero(~(outcome_total > 0.0))
    _core_require(
        bad.size == 0, f"shares need a positive outcome total; worlds {bad.tolist()} fail"
    )
    totals = {
        # Same reduction order as the library's treatment_total_contribution.
        "channel": (treatment.sum(axis=2) + indirect).sum(axis=1),
        "control": control.sum(axis=2).sum(axis=1),
        "baseline": intrinsic.sum(axis=1) + latent.sum(axis=2).sum(axis=1) + noise.sum(axis=1),
    }
    residual = np.abs(totals["channel"] + totals["control"] + totals["baseline"] - outcome_total)
    magnitude = (
        np.abs(outcome)
        + np.abs(treatment).sum(axis=2)
        + np.abs(indirect)
        + np.abs(control).sum(axis=2)
        + np.abs(intrinsic)
        + np.abs(latent).sum(axis=2)
        + np.abs(noise)
    ).sum(axis=1)
    bound = 2.0**-24 * magnitude
    failing = np.flatnonzero(residual > bound)
    _core_require(
        failing.size == 0,
        f"Σ parts − Σ outcome exceeds the float32 storage bound in worlds {failing.tolist()}",
    )
    shares = {part: total / outcome_total for part, total in totals.items()}
    library = pg.outcome_distributions(corpus, keep_series=False)
    grouped = shares["channel"] + shares["control"] + shares["baseline"]
    regrouping = np.abs(grouped - library.additive_share_total())
    split = np.abs(indirect).sum(axis=1) + np.abs(sources).sum(axis=(1, 2))
    tolerance = 1e-12 + 2.0**-24 * split / outcome_total
    failing = np.flatnonzero(~(regrouping <= tolerance))
    _core_require(
        failing.size == 0,
        f"three-part shares disagree with additive_share_total() in worlds {failing.tolist()}",
    )
    np.testing.assert_allclose(
        shares["channel"],
        library.quantities["treatment_total_contribution"].unit_share,
        rtol=1e-12,
        atol=0.0,
        err_msg="channel share must equal the treatment_total_contribution unit_share",
    )
    if show_closure:
        print(
            f"Share closure over {report.n_worlds} worlds: largest |Σ parts − Σ outcome| = "
            f"{residual.max():.3g} outcome units, at most {np.max(residual / bound):.2f}× "
            "its float32 storage bound."
        )
    return shares


# --- parameter grid ---------------------------------------------------------------


def _core_draws_per_cell(report):
    return int(report.corpus["diagnostics"]["draws_per_cell"])


def _core_cell_rows(report):
    """First stored world of every cell (one row per cell), in row order."""
    _, first = np.unique(np.asarray(report.corpus["cell_id"]), return_index=True)
    return np.sort(first)


def _core_pool(report, key, mask, rows=None):
    """Values of ``key`` where ``mask`` holds (broadcast over a jump/event axis), as float64."""
    array = np.asarray(report.corpus[key])
    mask = np.asarray(mask, dtype=bool)
    if rows is not None:
        array, mask = array[rows], mask[rows]
    if array.ndim == mask.ndim + 1:
        mask = np.broadcast_to(mask[:, None, :], array.shape)
    return array[mask].astype(np.float64)


def _core_softplus(x):
    """PyTensor's float64 softplus, branch for branch."""
    x = np.asarray(x, dtype=np.float64)
    with np.errstate(over="ignore"):
        return np.where(
            x < -37.0,
            np.exp(x),
            np.where(x < 18.0, np.log1p(np.exp(x)), np.where(x < 33.3, x + np.exp(-x), x)),
        )


def _core_n_bins(n):
    return int(min(10, max(3, math.isqrt(max(int(n), 1)))))


def _core_tolerance(precision, bound):
    eps, ulps = _CORE_PRECISION[precision]
    return ulps * eps * abs(float(bound)) + float(np.finfo(np.float64).tiny)


def _core_uniform(lo, hi, *, log=False):
    lo, hi = float(lo), float(hi)
    if lo == hi:
        return {"type": "point", "value": lo}
    return {"type": "loguniform" if log else "uniform", "lo": lo, "hi": hi}


def _core_atoms(masses, *, edges=None, slots=None):
    """Exact discrete law from {atom: rational mass}; a single atom is a point mass.

    ``slots`` optionally keeps the per-slot laws ({atom: mass} each, equal weight) of a
    pooled law that holds one event per slot, for the chance rate of its panel.
    """
    items = sorted((float(atom), Fraction(mass)) for atom, mass in masses.items() if mass > 0)
    _core_require(sum(mass for _, mass in items) == 1, "a discrete law's masses must sum to 1")
    if len(items) == 1:
        return {"type": "point", "value": items[0][0]}
    return {
        "type": "atoms",
        "atoms": np.asarray([atom for atom, _ in items]),
        "masses": [mass for _, mass in items],
        "edges": edges,
        "slots": slots if slots and len(slots) > 1 else None,
    }


def _core_binned(masses, edges):
    """Exact masses of a discrete law {atom: mass} per left-closed bin of ``edges``."""
    exact = [Fraction(0)] * (len(edges) - 1)
    atoms = np.asarray([float(atom) for atom in masses], dtype=np.float64)
    for index, mass in zip(_core_bin_index(atoms, edges).tolist(), masses.values()):
        exact[index] += Fraction(mass)
    return np.asarray([float(mass) for mass in exact])


def _core_discrete_uniform(lo, hi):
    lo, hi = int(lo), int(hi)
    return _core_atoms({value: Fraction(1, hi - lo + 1) for value in range(lo, hi + 1)})


def _core_floor_week_law(lo, hi, T):
    """Exact law of floor(u·T) for u ~ U(lo, hi) (a point mass when lo == hi).

    Masses are exact rationals over the float bounds. The support is the weeks the
    float64 product reaches, floor(lo·T) to floor(prev(hi)·T) (u < hi), and the end
    weeks absorb the sub-ulp slivers where the rational and float week boundaries
    differ (0.6 · 100 rounds up to 60), so the masses sum to exactly 1.
    """
    lo, hi = float(lo), float(hi)
    if lo == hi:
        return _core_atoms({math.floor(lo * T): Fraction(1)})
    a, b = Fraction(lo), Fraction(hi)
    first = math.floor(lo * T)
    last = math.floor(float(np.nextafter(hi, -np.inf)) * T)
    law = {}
    for week in range(first, last + 1):
        left = a if week == first else max(Fraction(week, T), a)
        right = b if week == last else min(Fraction(week + 1, T), b)
        if right > left:
            law[week] = (right - left) / (b - a)
    return _core_atoms(law)


def _core_jump_week_law(count, T):
    """Exact pooled law of jump weeks: event e is uniform on the e-th of `count` slots of [1, T)."""
    edges = [1 + e * (T - 1) // count for e in range(count + 1)]
    law, slots = {}, []
    for e in range(count):
        width = edges[e + 1] - edges[e]
        slot = {week: Fraction(1, width) for week in range(edges[e], edges[e + 1])}
        slots.append(slot)
        for week, mass in slot.items():
            law[week] = law.get(week, Fraction(0)) + mass / count
    return _core_atoms(law, slots=slots)


def _core_phase_law(cfg, role):
    """Exact pooled law of the flighting phase: uniform on 0..P-1 given P, P uniform."""
    p_lo, p_hi = (int(v) for v in getattr(cfg, f"{role}_flighting_period_weeks_range"))
    share = Fraction(1, p_hi - p_lo + 1)
    return _core_atoms(
        {
            phase: sum((share / p for p in range(max(p_lo, phase + 1), p_hi + 1)), Fraction(0))
            for phase in range(p_hi)
        }
    )


def _core_shock_start_law(cfg, T):
    """Exact pooled law of shock starts: DiscreteUniform(slot_lo, slot_hi − L) per slot."""
    count = int(cfg.n_treatment_shocks)
    len_lo, len_hi = (int(v) for v in cfg.treatment_shock_length_range)
    lengths = {length: Fraction(1, len_hi - len_lo + 1) for length in range(len_lo, len_hi + 1)}
    law, slots = {}, []
    for s in range(count):
        slot_lo, slot_hi = s * T // count, (s + 1) * T // count
        slot = {}
        if len_lo == len_hi and slot_hi - slot_lo == len_lo:
            slot[slot_lo] = Fraction(1)
        else:
            for length, p_length in lengths.items():
                starts = range(slot_lo, slot_hi - length + 1)
                for start in starts:
                    slot[start] = slot.get(start, Fraction(0)) + p_length / len(starts)
        slots.append(slot)
        for start, mass in slot.items():
            law[start] = law.get(start, Fraction(0)) + mass / count
    return _core_atoms(law, slots=slots)


def _core_integer_edges(lo, hi, n):
    """Half-integer bin edges over [lo, hi]: one bin per value up to 15 values, else grouped."""
    lo, hi = int(round(lo)), int(round(hi))
    span = hi - lo + 1
    width = 1 if span <= 15 else math.ceil(span / _core_n_bins(n))
    return lo - 0.5 + width * np.arange(math.ceil(span / width) + 1, dtype=np.float64)


def _core_auto_edges(values, *, integer, log, derived):
    """Histogram edges from the data alone (no configured law).

    A derived positive quantity spanning two decades or more gets a log axis; a
    primitive keeps a linear axis unless its law is known to be log-uniform.
    """
    lo, hi = float(values.min()), float(values.max())
    if integer:
        return _core_integer_edges(lo, hi, values.size), False
    if lo == hi:
        pad = 0.05 * abs(lo) if lo != 0.0 else 0.5
        return np.asarray([lo - pad, hi + pad]), False
    log = lo > 0.0 and (bool(log) or (derived and hi / lo >= 100.0))
    m = _core_n_bins(values.size)
    edges = np.geomspace(lo, hi, m + 1) if log else np.linspace(lo, hi, m + 1)
    edges[0], edges[-1] = lo, hi
    return edges, log


def _core_reference(law, values):
    """(edges, probs, support, display_edges) of a configured law at this panel's n."""
    kind, m = law["type"], _core_n_bins(values.size)
    if kind in ("uniform", "loguniform"):
        lo, hi = law["lo"], law["hi"]
        edges = np.geomspace(lo, hi, m + 1) if kind == "loguniform" else np.linspace(lo, hi, m + 1)
        edges[0], edges[-1] = lo, hi
        probs, support = np.full(m, 1.0 / m), (lo, hi)
    elif kind == "halfnormal":
        edges = stats.halfnorm.ppf(np.linspace(0.0, 1.0, m + 1), scale=law["sigma"])
        edges[0] = 0.0
        probs, support = np.full(m, 1.0 / m), (0.0, np.inf)
    elif kind == "beta":
        edges = stats.beta.ppf(np.linspace(0.0, 1.0, m + 1), law["a"], law["b"])
        edges[0], edges[-1] = 0.0, 1.0
        probs, support = np.full(m, 1.0 / m), (0.0, 1.0)
    elif kind == "atoms":
        atoms, edges = law["atoms"], law["edges"]
        if edges is None:
            edges = _core_integer_edges(atoms[0], atoms[-1], values.size)
            support = (atoms[0] - 0.5, atoms[-1] + 0.5)
        else:
            support = (float(edges[0]), float(edges[-1]))
        probs = _core_binned(dict(zip(atoms.tolist(), law["masses"])), edges)
    else:
        raise ValueError(f"unknown law type {kind!r}")
    display = np.asarray(edges, dtype=np.float64).copy()
    if not np.isfinite(display[-1]):
        tail = float(stats.halfnorm.ppf(1.0 - 0.2 / m, scale=law["sigma"]))
        display[-1] = max(float(values.max()), tail)
    return np.asarray(edges, dtype=np.float64), probs, support, display


def _core_outside(values, law, precision):
    """Values outside the configured support, beyond a few-ulp tolerance."""
    kind = law["type"]
    if kind == "atoms":
        return int(np.count_nonzero(~np.isin(values, law["atoms"])))
    lo, hi = {
        "uniform": (law.get("lo"), law.get("hi")),
        "loguniform": (law.get("lo"), law.get("hi")),
        "halfnormal": (0.0, np.inf),
        "beta": (0.0, 1.0),
    }[kind]
    below = values < lo - _core_tolerance(precision, lo)
    if np.isfinite(hi):
        above = values > hi + _core_tolerance(precision, hi)
    else:
        above = np.zeros_like(below)
    return int(np.count_nonzero(below | above))


def _core_hist(
    report,
    panels,
    *,
    group,
    title,
    unit,
    values,
    covers,
    color,
    law=None,
    reason="configured law unavailable",
    derived=None,
    cell=False,
    precision="float64",
    log=False,
    integer=False,
    expected_n=None,
    idle="no values",
):
    """Append one histogram panel (or a constant check / no-carrier entry) to ``panels``.

    ``idle`` says why a panel can be empty: "schedule" (no active input carries the
    schedule component, so the library drew nothing), "texture" (drawn for every
    input, carried by none) or "family" (drawn for every active channel, used by none
    because no channel selected the family).
    """
    values = np.asarray(values, dtype=np.float64).ravel()
    if expected_n is not None:
        _core_require(
            values.size == int(expected_n),
            f"{title}: pooled {values.size} values, expected {int(expected_n)} "
            "(one per carried flag)",
        )
    _core_require(bool(np.isfinite(values).all()), f"{title}: values must be finite")
    panel = {
        "group": group,
        "title": title,
        "unit": unit,
        "values": values,
        "n": int(values.size),
        "covers": set(covers),
        "color": color,
        "cell": cell,
        "log": bool(log),
        "probs": None,
        "support": None,
        "envelope": False,
        "note": "",
        "idle": idle,
    }
    if values.size == 0:
        panel["kind"] = "no carriers"
    elif derived is not None or law is None:
        edges, used_log = _core_auto_edges(
            values, integer=integer, log=log, derived=derived is not None
        )
        panel.update(
            kind="derived" if derived is not None else "unavailable",
            note=f"derived ({derived}): no band" if derived is not None else reason,
            edges=edges,
            display_edges=edges,
            log=used_log,
        )
    elif law["type"] == "point":
        tolerance = _core_tolerance(precision, law["value"])
        off = int(np.count_nonzero(np.abs(values - law["value"]) > tolerance))
        _core_require(
            off == 0,
            f"{title}: {off} of {values.size} values differ from the configured point mass "
            f"{law['value']:g}",
        )
        panel.update(kind="constant", point=law["value"])
    else:
        outside = _core_outside(values, law, precision)
        _core_require(
            outside == 0,
            f"{title}: {outside} of {values.size} values lie outside the configured support",
        )
        edges, probs, support, display = _core_reference(law, values)
        slots = law.get("slots")
        panel.update(
            kind="reference",
            edges=edges,
            display_edges=display,
            probs=probs,
            slot_probs=None if slots is None else [_core_binned(s, edges) for s in slots],
            support=support,
            log=law["type"] == "loguniform" or bool(log),
            envelope=not (cell and _core_draws_per_cell(report) > 1),
        )
    panels.append(panel)


def _core_bars(
    report,
    panels,
    *,
    group,
    title,
    unit,
    labels,
    counts,
    probs,
    n,
    covers,
    color,
    bernoulli=False,
    derived=None,
    spread=None,
):
    """Append one categorical bar panel; checks counts against probability-0/1 categories."""
    counts = np.asarray(counts)
    if probs is not None:
        probs = np.asarray(probs, dtype=np.float64)
        impossible = (probs == 0.0) & (counts > 0)
        _core_require(
            not impossible.any(),
            f"{title}: categories {[labels[i] for i in np.flatnonzero(impossible)]} have "
            "configured probability 0 but were drawn",
        )
        if bernoulli:
            certain = (probs == 1.0) & (counts != n)
            _core_require(
                not certain.any(),
                f"{title}: categories {[labels[i] for i in np.flatnonzero(certain)]} have "
                "configured probability 1 but were not drawn for every input",
            )
    note = "configured law unavailable" if derived is None else f"derived ({derived}): no band"
    panels.append(
        {
            "kind": "bars",
            "group": group,
            "title": title,
            "unit": unit,
            "labels": list(labels),
            "counts": counts,
            "probs": probs,
            "n": int(n),
            "covers": set(covers),
            "color": color,
            "envelope": probs is not None and _core_draws_per_cell(report) == 1,
            "bernoulli": bool(bernoulli),
            "note": note,
            "spread": spread,
        }
    )


def _core_structure_panels(report, cfg, panels):
    corpus, rows = report.corpus, _core_cell_rows(report)
    group = "structure"
    probs = None if cfg is None else cfg.trajectory_inclusion_probs()
    for role in TRAJECTORY_INPUTS:
        key = f"{role}_components"
        if key not in corpus:
            continue
        active = active_mask(report, role)[rows]
        flags = np.asarray(corpus[key])[rows][active]
        _core_bars(
            report,
            panels,
            group=group,
            title=f"{_CORE_NOUN[role].capitalize()}s carrying each component",
            unit=f"active {_CORE_NOUN[role]}s",
            labels=TRAJECTORY_COMPONENTS,
            counts=flags.sum(axis=0).astype(np.int64),
            probs=None if probs is None else [probs[role][c] for c in TRAJECTORY_COMPONENTS],
            n=int(active.sum()),
            covers={key},
            color=_CORE_COLORS[role],
            bernoulli=True,
        )
    active_t = active_mask(report, "treatment")[rows]
    for key, keys, prob_field, title in (
        ("carryover_family", CARRYOVER_FAMILY_KEYS, "carryover_family_probs", "Carryover family"),
        ("sat_family", SATURATION_FAMILY_KEYS, "saturation_family_probs", "Saturation family"),
    ):
        if key not in corpus:
            continue
        family = np.asarray(corpus[key])[rows][active_t].astype(np.int64)
        counts = np.bincount(family, minlength=len(keys))
        _core_require(counts.size == len(keys), f"{key} holds a family id outside {list(keys)}")
        _core_bars(
            report,
            panels,
            group=group,
            title=title,
            unit="active channels",
            labels=keys,
            counts=counts,
            probs=None if cfg is None else [float(getattr(cfg, prob_field)[k]) for k in keys],
            n=int(active_t.sum()),
            covers={key},
            color=_CORE_COLORS["treatment"],
        )
    law = None if cfg is None else _core_discrete_uniform(*cfg.n_latent_active_range_effective)
    _core_hist(
        report,
        panels,
        group=group,
        title="Active latent factors",
        unit="count per cell",
        values=np.asarray(corpus["n_latent_active"])[rows],
        law=law,
        covers={"n_latent_active"},
        color=_CORE_COLORS["world"],
        cell=True,
        integer=True,
    )
    panels.append(
        {
            "kind": "elsewhere",
            "group": group,
            "title": "active channel and control counts",
            "note": "shown in the count grid",
            "covers": {"n_treatments_active", "n_covariates_active"},
        }
    )
    layout = SlotLayout(
        n_treatments=corpus["treatment_active_mask"].shape[1],
        n_covariates=corpus["covariate_active_mask"].shape[1],
        n_latent=corpus["latent_active_mask"].shape[1],
    )
    g = np.asarray(corpus["g"])[rows].astype(np.int64)
    per_cell = np.stack(
        [g[:, layout.slices[edge]].sum(axis=1) for edge in layout.edge_types], axis=1
    )
    _core_bars(
        report,
        panels,
        group=group,
        title="Edges per cell by type (mean; whiskers min–max)",
        unit="edges per cell",
        labels=[_CORE_EDGE_LABELS[edge] for edge in layout.edge_types],
        counts=per_cell.mean(axis=0),
        probs=None,
        n=len(rows),
        covers={"g"},
        color=_CORE_COLORS["world"],
        derived="active counts and edge budgets",
        spread=(per_cell.min(axis=0), per_cell.max(axis=0)),
    )


def _core_walk_panels(report, cfg, panels):
    """Walk and noise scales (walk block): primitives against their laws, products derived."""
    if not report.walk:
        return
    corpus, rows = report.corpus, _core_cell_rows(report)
    relative = corpus["diagnostics"]["walk_parameters"]["outcome_std_mode"] == "relative"
    owners = {
        "treatment": (active_mask(report, "treatment"), _CORE_COLORS["treatment"]),
        "covariate": (active_mask(report, "covariate"), _CORE_COLORS["covariate"]),
        "world": (np.ones(report.n_worlds, dtype=bool), _CORE_COLORS["world"]),
        "latent": (np.asarray(corpus["latent_active_mask"]) == 1, _CORE_COLORS["world"]),
    }

    def add(key, title, unit, owner, law=None, derived=None, cell=False):
        mask, color = owners[owner]
        _core_hist(
            report,
            panels,
            group="walk",
            title=title,
            unit=unit,
            values=_core_pool(report, key, mask, rows if cell else None),
            law=None if cfg is None or law is None else law(),
            derived=derived,
            covers={key},
            color=color,
            cell=cell,
        )

    def uniform(field):
        return lambda: _core_uniform(*getattr(cfg, field))

    def halfnormal(field):
        return lambda: {"type": "halfnormal", "sigma": float(getattr(cfg, field))}

    def smoothness():
        return {
            "type": "beta",
            "a": float(cfg.rw_smoothness_alpha),
            "b": float(cfg.rw_smoothness_beta),
        }

    kernel = "share of max kernel"
    add(
        "walk_treatment_mean",
        "Channel walk mean (pre-softplus)",
        "drive units",
        "treatment",
        uniform("rw_positive_mean_range"),
    )
    add(
        "walk_treatment_std_relative",
        "Channel walk std, relative",
        "× channel level",
        "treatment",
        uniform("rw_treatment_std_range"),
    )
    add(
        "walk_treatment_std",
        "Channel walk std, absolute",
        "channel units",
        "treatment",
        derived="relative × level",
    )
    add(
        "walk_treatment_smoothness",
        "Channel walk smoothness",
        kernel,
        "treatment",
        smoothness,
        cell=True,
    )
    add(
        "walk_covariate_mean",
        "Control walk mean",
        "control units",
        "covariate",
        uniform("rw_covariate_mean_range"),
    )
    add(
        "walk_covariate_std",
        "Control walk std",
        "control units",
        "covariate",
        halfnormal("rw_std_sigma"),
    )
    add(
        "walk_covariate_smoothness",
        "Control walk smoothness",
        kernel,
        "covariate",
        smoothness,
        cell=True,
    )
    add(
        "walk_baseline_mean",
        "Baseline walk mean",
        "outcome units",
        "world",
        uniform("rw_baseline_mean_range"),
    )
    if relative:
        add(
            "walk_baseline_std_relative",
            "Baseline walk std, relative",
            "× effect scale",
            "world",
            uniform("rw_baseline_std_range"),
        )
        add(
            "walk_baseline_std",
            "Baseline walk std, absolute",
            "outcome units",
            "world",
            derived="relative × effect scale",
        )
    else:
        add(
            "walk_baseline_std",
            "Baseline walk std",
            "outcome units",
            "world",
            halfnormal("rw_baseline_std_sigma_effective"),
        )
    add(
        "walk_baseline_smoothness",
        "Baseline walk smoothness",
        kernel,
        "world",
        smoothness,
        cell=True,
    )
    if relative:
        add(
            "outcome_noise_std_relative",
            "Outcome noise std, relative",
            "× effect scale",
            "world",
            uniform("rw_outcome_std_range"),
        )
        add(
            "outcome_noise_std",
            "Outcome noise std, absolute",
            "outcome units",
            "world",
            derived="relative × effect scale",
        )
    else:
        add(
            "outcome_noise_std",
            "Outcome noise std",
            "outcome units",
            "world",
            halfnormal("rw_outcome_std_sigma"),
        )
    add("walk_latent_smoothness", "Latent walk smoothness", kernel, "latent", smoothness, cell=True)


def _core_family_mask(report, key, keys, family, active):
    """Active channels whose drawn family is ``family``; every active channel if not stored."""
    if key not in report.corpus:
        return active
    family_ids = np.asarray(report.corpus[key])
    return active & (family_ids == keys.index(family))


def _core_mechanism_panels(report, cfg, panels):
    corpus, group = report.corpus, "mechanism"
    blue, orange = _CORE_COLORS["treatment"], _CORE_COLORS["covariate"]
    active_t, active_c = active_mask(report, "treatment"), active_mask(report, "covariate")
    conditioned = cfg is not None and bool(cfg.prior_conditioning)
    conditioned_reason = "configured law unavailable: conditioned per cell (prior_conditioning)"
    # A shape draw generates data only for the channels whose family uses it; the
    # other channels' draws exist but do nothing, so they are not pooled.
    for key, title, unit, range_field, family in (
        (
            "carryover_alpha",
            "Carryover alpha",
            "decay per week",
            "carryover_alpha_range",
            "geometric",
        ),
        ("weibull_lam", "Weibull carryover lambda", "weeks", "weibull_lam_range", "weibull"),
        ("weibull_k", "Weibull carryover k", "shape", "weibull_k_range", "weibull"),
    ):
        unavailable = key == "carryover_alpha" and conditioned
        users = _core_family_mask(
            report, "carryover_family", CARRYOVER_FAMILY_KEYS, family, active_t
        )
        _core_hist(
            report,
            panels,
            group=group,
            title=title,
            unit=unit,
            values=_core_pool(report, key, users),
            law=None if cfg is None or unavailable else _core_uniform(*getattr(cfg, range_field)),
            reason=conditioned_reason if unavailable else "configured law unavailable",
            covers={key},
            color=blue,
            precision="float32",
            idle="family",
        )
    # The stored recipe is the echo the sidecar check verified, so it also serves
    # without a sidecar for presentation choices (log axis, derived coefficients).
    recipe = corpus["diagnostics"].get("mechanism_priors", {})
    mm_log = recipe.get("mm_scale_prior") == "log_uniform"
    for key, (family, parameter) in MECHANISM_PRIOR_FIELDS.items():
        if key not in corpus:
            continue
        unavailable = key == "hill_slope" and conditioned
        log = key == "mm_kappa_mult" and mm_log
        law = None
        if cfg is not None and not unavailable:
            law = _core_uniform(*cfg.saturation_prior_ranges[family][parameter], log=log)
        users = _core_family_mask(report, "sat_family", SATURATION_FAMILY_KEYS, family, active_t)
        _core_hist(
            report,
            panels,
            group=group,
            title=_CORE_SHAPE_TITLES[key],
            unit="× saturation anchor" if parameter == "kappa_mult" else "shape",
            values=_core_pool(report, key, users),
            law=law,
            reason=conditioned_reason if unavailable else "configured law unavailable",
            covers={key},
            color=blue,
            log=log,
            idle="family",
        )
    for key, role, active, color, title, unit, range_field, how in (
        (
            "beta",
            "treatment",
            active_t,
            blue,
            "Channel effect beta",
            "outcome per response unit",
            "beta_additive_range",
            "target ÷ reference response",
        ),
        (
            "rho_zy",
            "covariate",
            active_c,
            orange,
            "Control effect rho",
            "outcome per control unit",
            "zy_coeff_range",
            "target ÷ reference scale",
        ),
    ):
        if key not in corpus:
            continue
        derived = how if recipe.get(f"{role}_reference_contribution_range") is not None else None
        _core_hist(
            report,
            panels,
            group=group,
            title=title,
            unit=unit,
            values=_core_pool(report, key, active),
            law=None if cfg is None or derived else _core_uniform(*getattr(cfg, range_field)),
            derived=derived,
            covers={key},
            color=color,
        )
    anchor = "saturation_scale"
    if "mechanism_saturation_scale" in corpus:
        anchor = "mechanism_saturation_scale"
    _core_hist(
        report,
        panels,
        group=group,
        title="Saturation anchor",
        unit="channel units",
        values=_core_pool(report, anchor, active_t),
        derived="function of the channel's draws",
        covers={"saturation_scale", "mechanism_saturation_scale"},
        color=blue,
    )
    _core_hist(
        report,
        panels,
        group=group,
        title="Channel level",
        unit="channel units",
        values=_core_pool(report, "treatment_level", active_t),
        derived="softplus of the walk mean",
        covers={"treatment_level"},
        color=blue,
    )
    for role, active, color in (("treatment", active_t, blue), ("covariate", active_c, orange)):
        key = f"{role}_reference_contribution"
        if key not in corpus:
            continue
        range_field = f"{role}_reference_contribution_range"
        _core_hist(
            report,
            panels,
            group=group,
            title=f"{_CORE_NOUN[role].capitalize()} reference contribution",
            unit="outcome units",
            values=_core_pool(report, key, active),
            law=None if cfg is None else _core_uniform(*getattr(cfg, range_field)),
            covers={key},
            color=color,
        )
    if "treatment_reference_input" in corpus:
        for key, title, unit, how in (
            (
                "treatment_reference_input",
                "Channel reference input",
                "channel units",
                "multiplier × anchor",
            ),
            (
                "treatment_reference_response",
                "Channel reference response",
                "response units",
                "response at the reference",
            ),
        ):
            _core_hist(
                report,
                panels,
                group=group,
                title=title,
                unit=unit,
                values=_core_pool(report, key, active_t),
                derived=how,
                covers={key},
                color=blue,
            )
    if "covariate_reference_input" in corpus:
        scale = None
        if cfg is not None:
            scale = {"type": "point", "value": float(cfg.covariate_reference_scale)}
        _core_hist(
            report,
            panels,
            group=group,
            title="Control reference input",
            unit="control units",
            values=_core_pool(report, "covariate_reference_input", active_c),
            law=scale,
            covers={"covariate_reference_input"},
            color=orange,
        )
    if cfg is None:
        strength = None
    elif cfg.confounding_strength_range is None:
        strength = {"type": "point", "value": 0.0}
    else:
        strength = _core_uniform(*cfg.confounding_strength_range)
    _core_hist(
        report,
        panels,
        group=group,
        title="Confounding strength",
        unit="innovation correlation",
        values=np.asarray(corpus["confounding_strength"]),
        law=strength,
        covers={"confounding_strength"},
        color=_CORE_COLORS["world"],
        precision="float32",
    )


def _core_trajectory_panels(report, cfg, role, panels):
    corpus = report.corpus
    if f"{role}_components" not in corpus:
        return
    noun, color = _CORE_NOUN[role], _CORE_COLORS[role]
    prefix, T = f"trajectory_{role}_", report.T
    level_unit = f"{noun} units"
    scale_name = "channel level" if role == "treatment" else "control walk std"

    def add(component, field, title, unit, values, law, **extra):
        carried = carried_mask(report, role, component)
        stored = np.asarray(corpus[f"{prefix}{component}_{field}"])
        events = stored.shape[1] if stored.ndim == 3 else 1
        _core_hist(
            report,
            panels,
            group=role,
            title=f"{noun.capitalize()} {title}",
            unit=unit,
            values=values,
            law=law,
            color=color,
            expected_n=int(carried.sum()) * events,
            idle="texture" if component in TEXTURE_COMPONENTS else "schedule",
            **extra,
        )

    def leaf(component, field):
        carried = carried_mask(report, role, component)
        return _core_pool(report, f"{prefix}{component}_{field}", carried)

    def configured_law(build, *args, **kwargs):
        """``build(*args)`` on the configured ranges, or None without a config."""
        return None if cfg is None else build(*args, **kwargs)

    def configured_range(field):
        """The role's configured ``field``; empty without a config (``law`` returns None)."""
        return getattr(cfg, f"{role}_{field}") if cfg is not None else ()

    for component, field, name in (
        ("hf", "sigma", "hf noise sigma"),
        ("pulse", "amp", "pulse amplitude"),
    ):
        carried = carried_mask(report, role, component)
        key = f"{prefix}{component}_{field}"
        effective = _core_pool(report, key, carried)
        add(
            component,
            field,
            f"{name} (effective)",
            level_unit,
            effective,
            None,
            derived=f"multiplier × {scale_name}",
            covers={key},
        )
        if report.walk and effective.size:
            if role == "treatment":
                scale = _core_softplus(corpus["walk_treatment_mean"])[carried]
            else:
                scale = np.asarray(corpus["walk_covariate_std"], dtype=np.float64)[carried]
            with np.errstate(divide="ignore", invalid="ignore"):
                multiplier = effective / scale
            add(
                component,
                field,
                f"{name} multiplier",
                f"× {scale_name}",
                multiplier,
                configured_law(_core_uniform, *configured_range(f"{component}_{field}_range")),
                covers=set(),
                precision="ratio",
            )
    add(
        "pulse",
        "prob",
        "pulse probability",
        "per week",
        leaf("pulse", "prob"),
        configured_law(_core_uniform, *configured_range("pulse_prob_range")),
        covers={f"{prefix}pulse_prob"},
    )
    for gate, field in (("onset", "start"), ("offset", "stop")):
        add(
            gate,
            field,
            f"{gate} week",
            "reported week",
            leaf(gate, field),
            configured_law(_core_floor_week_law, *configured_range(f"{gate}_frac_range"), T),
            covers={f"{prefix}{gate}_{field}"},
            integer=True,
        )
    period = leaf("flighting", "period")
    add(
        "flighting",
        "period",
        "flighting period",
        "weeks",
        period,
        configured_law(_core_discrete_uniform, *configured_range("flighting_period_weeks_range")),
        covers={f"{prefix}flighting_period"},
        integer=True,
    )
    duty_law = None
    if cfg is not None:
        masses = {}
        for (p, w), mass in _core_flighting_law(cfg, role).items():
            masses[w / p] = masses.get(w / p, Fraction(0)) + mass
        duty_law = _core_atoms(masses, edges=np.arange(11) / 10.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        duty = leaf("flighting", "on_weeks") / period
    add(
        "flighting",
        "on_weeks",
        "flighting W/P",
        "on-weeks ÷ period (duty observed only through on_weeks)",
        duty,
        duty_law,
        covers={f"{prefix}flighting_on_weeks"},
    )
    add(
        "flighting",
        "phase",
        "flighting phase",
        "weeks",
        leaf("flighting", "phase"),
        configured_law(_core_phase_law, cfg, role),
        covers={f"{prefix}flighting_phase"},
        integer=True,
    )
    jump_count = corpus[f"{prefix}level_jump_week"].shape[1]
    if cfg is not None:
        configured_count = int(configured_range("level_jump_count"))
        _core_require(
            jump_count == configured_count,
            f"{prefix}level_jump_week has {jump_count} events per input, "
            f"the config says {configured_count}",
        )
    add(
        "level_jump",
        "week",
        "level-jump week",
        "reported week",
        leaf("level_jump", "week"),
        configured_law(_core_jump_week_law, jump_count, T),
        covers={f"{prefix}level_jump_week"},
        integer=True,
    )
    if role == "treatment":
        add(
            "level_jump",
            "factor",
            "level-jump factor",
            "level multiplier",
            leaf("level_jump", "factor"),
            configured_law(_core_uniform, *configured_range("level_jump_factor_range"), log=True),
            covers={f"{prefix}level_jump_factor", f"{prefix}level_jump_log_factor"},
            log=True,
        )
    else:
        add(
            "level_jump",
            "size",
            "level-jump size",
            level_unit,
            leaf("level_jump", "size"),
            configured_law(_core_uniform, *configured_range("level_jump_size_range")),
            covers={f"{prefix}level_jump_size"},
        )
    amplitude_unit = "log-level" if role == "treatment" else level_unit
    for field, title, unit, range_field in (
        ("amplitude", "seasonal amplitude", amplitude_unit, "seasonal_amplitude_range"),
        ("period", "seasonal period", "weeks", "seasonal_period_weeks_range"),
    ):
        add(
            "seasonal",
            field,
            title,
            unit,
            leaf("seasonal", field),
            configured_law(_core_uniform, *configured_range(range_field)),
            covers={f"{prefix}seasonal_{field}"},
        )
    add(
        "seasonal",
        "phase",
        "seasonal phase",
        "radians",
        leaf("seasonal", "phase"),
        configured_law(_core_uniform, 0.0, 2.0 * np.pi),
        covers={f"{prefix}seasonal_phase"},
    )
    if role == "treatment":
        trend_unit, trend_range = "log-level change over the window", "trend_log_change_range"
    else:
        trend_unit, trend_range = f"{level_unit} over the window", "trend_change_range"
    add(
        "trend",
        "change",
        "trend change",
        trend_unit,
        leaf("trend", "change"),
        configured_law(_core_uniform, *configured_range(trend_range)),
        covers={f"{prefix}trend_change"},
    )


def _core_event_panels(report, cfg, panels):
    corpus, group, blue = report.corpus, "events", _CORE_COLORS["treatment"]
    shock_keys = {
        "treatment_shock_index",
        "treatment_shock_start",
        "treatment_shock_length",
        "treatment_shock_level_multiplier",
        "treatment_shock_level",
    }
    n_events = corpus["treatment_shock_index"].shape[1]
    if n_events == 0:
        panels.append(
            {
                "kind": "no events",
                "group": group,
                "title": "channel shocks",
                "note": "no shock events (n_treatment_shocks = 0)",
                "covers": shock_keys,
            }
        )
    else:
        events = np.ones((report.n_worlds, n_events), dtype=bool)
        if cfg is None:
            length_law = multiplier_law = start_law = None
        else:
            length_law = _core_discrete_uniform(*cfg.treatment_shock_length_range)
            multiplier_law = _core_uniform(*cfg.treatment_shock_level_range)
            start_law = _core_shock_start_law(cfg, report.T)
        for key, title, unit, law, integer in (
            ("treatment_shock_start", "Shock start", "reported week", start_law, True),
            ("treatment_shock_length", "Shock length", "weeks", length_law, True),
            (
                "treatment_shock_level_multiplier",
                "Shock level multiplier",
                "× channel level",
                multiplier_law,
                False,
            ),
        ):
            _core_hist(
                report,
                panels,
                group=group,
                title=title,
                unit=unit,
                values=_core_pool(report, key, events),
                law=law,
                covers={key},
                color=blue,
                integer=integer,
                precision="float32",
            )
        for key, title, unit, how, integer in (
            (
                "treatment_shock_index",
                "Shocked channel slot",
                "slot",
                "uniform over direct channels",
                True,
            ),
            (
                "treatment_shock_level",
                "Shock held level",
                "channel units",
                "multiplier × level",
                False,
            ),
        ):
            _core_hist(
                report,
                panels,
                group=group,
                title=title,
                unit=unit,
                values=_core_pool(report, key, events),
                derived=how,
                covers={key},
                color=blue,
                integer=integer,
            )
    if "prior_cond" in corpus:
        rows = _core_cell_rows(report)
        intervals = np.asarray(corpus["prior_cond"], dtype=np.float64)[rows]
        spec = None if cfg is None else cfg.prior_cond_spec()
        for quantity in PRIOR_COND_QUANTITIES:
            width = intervals[:, PRIOR_COND_LAYOUT.index(f"{quantity}_width")]
            low = intervals[:, PRIOR_COND_LAYOUT.index(f"{quantity}_low")]
            label = quantity.replace("_", " ")
            _core_hist(
                report,
                panels,
                group=group,
                title=f"{label} interval width",
                unit="parameter units",
                values=width,
                law=None if spec is None else _core_uniform(*spec[quantity]["width_range"]),
                covers={"prior_cond"},
                color=_CORE_COLORS["world"],
                cell=True,
                precision="float32",
            )
            _core_hist(
                report,
                panels,
                group=group,
                title=f"{label} interval low",
                unit="parameter units",
                values=low,
                derived="drawn given the width",
                covers={"prior_cond"},
                color=_CORE_COLORS["world"],
                cell=True,
            )


def _core_classify(report):
    """Assert the completeness classification; return (present, absent) parameter keys.

    (1) top-level keys: pairwise-disjoint registry classes cover every present key and
    the core registry splits exactly into parameters and outputs; (2) diagnostics keys
    are all library keys; (3) the sidecar config universe is checked by load_report.
    """
    corpus = report.corpus
    classes = {
        "core": set(CORPUS_ARRAY_FIELDS),
        "trajectory structure": set(TRAJECTORY_ARRAY_FIELDS) - set(TRAJECTORY_PARAM_FIELDS),
        "trajectory parameters": set(TRAJECTORY_PARAM_FIELDS),
        "mechanism": set(MECHANISM_ARRAY_FIELDS),
        "mechanism reference": set(MECHANISM_REFERENCE_FIELDS),
        "walk": set(WALK_ARRAY_FIELDS),
        "metadata": set(_CORE_METADATA_KEYS),
    }
    names = list(classes)
    for i, first in enumerate(names):
        for second in names[i + 1 :]:
            overlap = sorted(classes[first] & classes[second])
            _core_require(not overlap, f"key classes {first} and {second} overlap on {overlap}")
    unclassified = sorted(set(corpus) - set().union(*classes.values()))
    _core_require(not unclassified, f"corpus keys outside every registry class: {unclassified}")
    split = _CORE_PARAMETER_KEYS | _CORE_OUTPUT_KEYS
    _core_require(
        split == classes["core"] and not _CORE_PARAMETER_KEYS & _CORE_OUTPUT_KEYS,
        "the report's split of CORPUS_ARRAY_FIELDS into parameters and outputs is out of "
        f"date: {sorted(classes['core'] ^ split)}",
    )
    unknown = sorted(set(corpus["diagnostics"]) - _CORE_DIAGNOSTICS_KEYS)
    _core_require(not unknown, f"diagnostics keys unknown to the report: {unknown}")
    parameters = (
        set(_CORE_PARAMETER_KEYS)
        | {f"{role}_components" for role in TRAJECTORY_INPUTS}
        | classes["trajectory parameters"]
        | classes["mechanism"]
        | classes["mechanism reference"]
        | classes["walk"]
        | {"prior_cond"}
    )
    return parameters & set(corpus), parameters - set(corpus)


def _core_unused_reason(diagnostics, label):
    """Why an absent parameter block was never drawn by its configuration, or None.

    None means the block was drawn but not saved ("not recorded").
    """
    if label == "trajectory schedule components" and "trajectory" not in diagnostics:
        # The library writes the trajectory block whenever a schedule component can occur.
        return "no schedule component could be included"
    if label == "prior conditioning":
        return "off"  # the library writes prior_cond whenever prior conditioning is on
    if label == "relative-mode baseline/noise draws":
        if diagnostics.get("signal", {}).get("outcome_std_mode") == "absolute":
            return "absolute outcome_std_mode draws absolute scales instead"
    if label in ("channel reference block", "control reference block"):
        role = "treatment" if label.startswith("channel") else "covariate"
        recipe = diagnostics.get("mechanism_priors")
        if recipe is None or recipe.get(f"{role}_reference_contribution_range") is None:
            return "no reference-contribution prior is configured"
    return None


def _core_trajectory_component(key):
    """The trajectory component a ``trajectory_{role}_{component}_{leaf}`` array belongs to."""
    for role in TRAJECTORY_INPUTS:
        prefix = f"trajectory_{role}_"
        if key.startswith(prefix):
            rest = key[len(prefix) :]
            return next(c for c in TRAJECTORY_COMPONENTS if rest.startswith(f"{c}_"))
    raise ValueError(f"{key} is not a trajectory parameter array")


def _core_not_recorded(absent, panels, diagnostics):
    components = {f"{role}_components" for role in TRAJECTORY_INPUTS}
    references = {
        role: {key for key in MECHANISM_REFERENCE_FIELDS if key.startswith(f"{role}_")}
        for role in TRAJECTORY_INPUTS
    }
    texture = {
        key
        for key in TRAJECTORY_PARAM_FIELDS
        if _core_trajectory_component(key) in TEXTURE_COMPONENTS
    }
    blocks = (
        ("trajectory schedule components", set(TRAJECTORY_PARAM_FIELDS) - texture),
        ("trajectory texture parameters and component flags", components | texture),
        ("mechanism block", set(MECHANISM_ARRAY_FIELDS)),
        ("channel reference block", references["treatment"]),
        ("control reference block", references["covariate"]),
        ("walk and noise block", set(WALK_ARRAY_FIELDS) - set(WALK_RELATIVE_MODE_FIELDS)),
        ("relative-mode baseline/noise draws", set(WALK_RELATIVE_MODE_FIELDS)),
        ("prior conditioning", {"prior_cond"}),
    )
    listed = set()
    for label, keys in blocks:
        missing = sorted(keys & absent)
        listed |= set(missing)
        if missing:
            reason = _core_unused_reason(diagnostics, label)
            panels.append(
                {
                    "kind": "not recorded" if reason is None else "not used",
                    "group": None,
                    "title": label,
                    "keys": missing,
                    "note": reason,
                    "covers": set(),
                }
            )
    _core_require(
        listed == absent, f"absent parameter arrays without a block: {sorted(absent - listed)}"
    )


def parameter_panels(report) -> list[dict]:
    """Every generation parameter as a panel spec, completeness classification asserted.

    Each dict has ``kind`` in {"reference", "derived", "unavailable", "bars", "constant",
    "no carriers", "no events", "elsewhere", "not recorded", "not used"}, a ``group``
    and ``covers`` (the corpus keys it accounts for). Per-input quantities pool active
    inputs that carry the component (count asserted against the flags); cell-level ones
    use one row per cell. Laws come from the rebuilt ``SCMPrior(**config)`` and every
    reference panel's values are asserted inside the configured support.
    """
    present, absent = _core_classify(report)
    cfg = _core_prior(report)
    panels = []
    _core_structure_panels(report, cfg, panels)
    _core_walk_panels(report, cfg, panels)
    _core_mechanism_panels(report, cfg, panels)
    for role in TRAJECTORY_INPUTS:
        _core_trajectory_panels(report, cfg, role, panels)
    _core_event_panels(report, cfg, panels)
    _core_not_recorded(absent, panels, report.corpus["diagnostics"])
    covered = set().union(*(panel["covers"] for panel in panels))
    _core_require(
        present <= covered,
        f"parameters present in the corpus but missing from the grid: {sorted(present - covered)}",
    )
    return panels


def _core_acceptance_text(report):
    rate = report.acceptance.get("rejection_rate")
    return "acceptance not recorded" if rate is None else f"acceptance {100.0 * (1.0 - rate):.1f}%"


def _core_draw_hist(ax, panel, draws_per_cell):
    """Draw one histogram panel; return k (bins outside) when an envelope is drawn, else None."""
    values, edges, display = panel["values"], panel["edges"], panel["display_edges"]
    counts = np.bincount(_core_bin_index(values, edges), minlength=len(edges) - 1)
    if panel["support"] is not None:
        left, right = panel["support"]
        ax.axvspan(left, min(right, display[-1]), color="#2ca02c", alpha=0.08, lw=0, zorder=0)
    edgecolors = ["white"] * counts.size
    k = None
    status = panel["note"]
    if panel["probs"] is not None:
        expected, lo, hi = sidak_envelope(panel["probs"], panel["n"])
        if panel["log"]:
            centers = np.sqrt(display[:-1] * display[1:])
        else:
            centers = (display[:-1] + display[1:]) / 2
        if panel["envelope"]:
            outside = (counts < lo) | (counts > hi)
            edgecolors = ["#c0392b" if out else "white" for out in outside]
            _core_envelope_marks(ax, centers, expected, lo, hi)
            k = int(outside.sum())
            status = f"{k} of {counts.size} bins outside"
        else:
            ax.scatter(centers, expected, s=12, color="black", zorder=4)
            status = f"no envelope: draws_per_cell = {draws_per_cell}"
    ax.bar(
        display[:-1],
        counts,
        width=np.diff(display),
        align="edge",
        color=panel["color"],
        edgecolor=edgecolors,
        linewidth=1.0,
        zorder=2,
    )
    if panel["log"]:
        ax.set_xscale("log")
    ax.set_title(f"{panel['title']} · n {panel['n']}\n{status}", fontsize=8)
    ax.set_xlabel(panel["unit"], fontsize=7)
    ax.set_ylabel("count", fontsize=7)
    ax.tick_params(labelsize=7)
    return k


def _core_draw_bar_panel(ax, panel, draws_per_cell):
    """Draw one categorical panel; return k when an envelope is drawn, else None."""
    if panel["envelope"]:
        k, _ = draw_reference_bars(
            ax,
            panel["counts"],
            panel["probs"],
            panel["n"],
            panel["labels"],
            color=panel["color"],
            title=panel["title"],
        )
        ax.set_ylabel(panel["unit"], fontsize=7)
        return k
    x = np.arange(len(panel["labels"]))
    ax.bar(x, panel["counts"], width=0.75, color=panel["color"], zorder=2)
    if panel["spread"] is not None:
        low, high = panel["spread"]
        ax.vlines(x, low, high, color="black", linewidth=1.0, zorder=3)
    if panel["probs"] is not None:
        ax.scatter(x, panel["n"] * panel["probs"], s=12, color="black", zorder=4)
        status = f"no envelope: draws_per_cell = {draws_per_cell}"
    else:
        status = panel["note"]
    _core_category_ticks(ax, x, panel["labels"])
    ax.set_ylabel(panel["unit"], fontsize=7)
    ax.set_title(f"{panel['title']} · n {panel['n']}\n{status}", fontsize=8)
    return None


def plot_parameter_grid(report) -> None:
    """Every generation parameter against its configured law, one figure per group."""
    panels = parameter_panels(report)
    draws_per_cell = _core_draws_per_cell(report)
    print(
        "How to read the parameter grid: bars count worlds, inputs or events per bin; black "
        "dots are the counts expected under the configured law before the realism filter "
        f"({_core_acceptance_text(report)}); black whiskers are a Šidák family-wise envelope "
        "(exact binomial quantiles) that holds the whole panel with probability at least "
        f"{ENVELOPE_LEVEL:.0%} at the panel's n, and 'k of m bins outside' counts red-edged "
        "bars that leave it. Chance alone does that in at most one reference panel in ten, "
        "fewer at small n; the expected number is printed below the figures. Green shading "
        "is the configured support. Derived panels are functions of other draws and carry "
        "no band. Per-input panels pool the active inputs that carry the component; "
        "cell-level panels use one row per cell."
    )
    if report.config is None:
        print("Configured law unavailable: no sidecar config, so every panel is a histogram only.")
    envelopes = flagged = certain = 0
    chance = 0.0
    for group, heading in _CORE_GROUPS.items():
        drawable = [p for p in panels if p.get("group") == group and p["kind"] in _CORE_DRAWABLE]
        if not drawable:
            continue
        ncols = min(4, len(drawable))
        nrows = math.ceil(len(drawable) / ncols)
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(3.8 * ncols, 3.0 * nrows), squeeze=False, layout="constrained"
        )
        for ax, panel in zip(axes.flat, drawable):
            draw = _core_draw_bar_panel if panel["kind"] == "bars" else _core_draw_hist
            k = draw(ax, panel, draws_per_cell)
            if k is not None:
                probs = np.asarray(panel["probs"], dtype=np.float64)
                if np.all((probs == 0.0) | (probs == 1.0)):
                    certain += 1
                    continue
                envelopes += 1
                flagged += k > 0
                chance += _core_family_rate(
                    probs,
                    panel["n"],
                    bernoulli=panel.get("bernoulli"),
                    slots=panel.get("slot_probs"),
                )
        for ax in list(axes.flat)[len(drawable) :]:
            ax.remove()
        fig.suptitle(heading, fontsize=11)
        plt.show()
        plt.close(fig)
    if envelopes:
        print(
            f"Panels with an envelope: {envelopes}; with at least one bin outside: {flagged}; "
            f"expected by chance alone: {chance:.1f} (each panel's probability of leaving its "
            "envelope when the generator follows its law, summed; at most "
            f"{1.0 - ENVELOPE_LEVEL:.2f} per panel)."
        )
    if certain:
        print(
            f"{certain} more panels have only certain categories (probability 0 or 1), so "
            "they cannot leave their envelope and are not counted."
        )
    if draws_per_cell > 1:
        print(f"Cell-level panels carry no envelope because draws_per_cell = {draws_per_cell}.")
    for panel in panels:
        if panel["kind"] == "constant":
            print(
                f"Constant check: {panel['title']} equals the configured point mass "
                f"{panel['point']:g} {panel['unit']} for all {panel['n']} values."
            )
    reasons = (
        ("schedule", "Not drawn — no active input carries the component"),
        ("texture", "Drawn for every input but carried by no active input"),
        ("family", "Drawn for every active channel but used by none — family not selected"),
        ("no values", "No values to plot"),
    )
    for reason, text in reasons:
        idle = [p["title"] for p in panels if p["kind"] == "no carriers" and p["idle"] == reason]
        if idle:
            print(f"{text}: " + "; ".join(idle) + ".")
    for panel in panels:
        if panel["kind"] == "no events":
            print(f"{panel['title'].capitalize()}: {panel['note']}.")
    for panel in panels:
        if panel["kind"] == "not used":
            print(f"Not used by this configuration: {panel['title']} ({panel['note']}).")
        elif panel["kind"] == "not recorded":
            keys = panel["keys"]
            shown = ", ".join(keys)
            if len(keys) > 6:
                shown = ", ".join(keys[:4]) + f", … ({len(keys)} arrays)"
            print(f"Not recorded: {panel['title']} — {shown}.")


# --- count grid -------------------------------------------------------------------


def _core_count_matrix(t, c, t_axis, c_axis):
    grid = np.zeros((len(t_axis), len(c_axis)), dtype=np.int64)
    row = {value: i for i, value in enumerate(t_axis)}
    col = {value: j for j, value in enumerate(c_axis)}
    for a, b in zip(t.tolist(), c.tolist()):
        if a in row and b in col:
            grid[row[a], col[b]] += 1
    return grid


def _core_spearman_text(name, x, y):
    if np.unique(x).size < 2 or np.unique(y).size < 2:
        return f"{name}: n/a (constant axis)"
    return f"{name} = {float(stats.spearmanr(x, y)[0]):+.3f}"


def _core_marginal(ax, counts, expected, envelope, *, vertical, color):
    positions = np.arange(counts.size)
    if vertical:
        ax.bar(positions, counts, width=0.75, color=color, zorder=2)
    else:
        ax.barh(positions, counts, height=0.75, color=color, zorder=2)
    if expected is not None:
        dots = (positions, expected) if vertical else (expected, positions)
        ax.scatter(*dots, s=14, color="black", zorder=4)
    if envelope is not None:
        lo, hi = envelope
        lines = ax.vlines if vertical else ax.hlines
        lines(positions, lo, hi, color="black", linewidth=1.0, zorder=3)
    ax.tick_params(labelsize=7)


def plot_count_grid(report) -> None:
    """Heatmap of active channel × control counts with both marginals and Spearman ρ."""
    corpus = report.corpus
    t_all = np.asarray(corpus["n_treatments_active"], dtype=np.int64)
    c_all = np.asarray(corpus["n_covariates_active"], dtype=np.int64)
    _core_require(
        np.array_equal(t_all, active_mask(report, "treatment").sum(axis=1))
        and np.array_equal(c_all, active_mask(report, "covariate").sum(axis=1)),
        "n_treatments_active / n_covariates_active must equal the active-mask counts",
    )
    rows = _core_cell_rows(report)
    draws_per_cell = _core_draws_per_cell(report)
    cfg = _core_prior(report)
    coverage = corpus["diagnostics"].get("active_count_coverage")
    if coverage is not None:
        t_axis = list(coverage["n_treatments_active"])
        c_axis = list(coverage["n_covariates_active"])
    elif cfg is not None:
        t_axis, c_axis = (list(axis) for axis in cfg.active_count_grid)
    else:
        t_axis = list(range(int(t_all.min()), int(t_all.max()) + 1))
        c_axis = list(range(int(c_all.min()), int(c_all.max()) + 1))
    worlds = _core_count_matrix(t_all, c_all, t_axis, c_axis)
    cells = _core_count_matrix(t_all[rows], c_all[rows], t_axis, c_axis)
    _core_require(
        worlds.sum() == report.n_worlds, "every world's active counts must lie on the grid"
    )
    if coverage is not None:
        np.testing.assert_array_equal(worlds, np.asarray(coverage["n_worlds"], dtype=np.int64))
        np.testing.assert_array_equal(cells, np.asarray(coverage["n_cells"], dtype=np.int64))
    weights = _core_design_weights(report)
    if weights is not None:
        grid_t, grid_c = np.meshgrid(t_axis, c_axis, indexing="ij")
        x = np.repeat(grid_t.ravel(), weights.ravel())
        y = np.repeat(grid_c.ravel(), weights.ravel())
        rho_text = _core_spearman_text("design ρ_W", x, y)
    else:
        rho_text = _core_spearman_text("realized ρ (one row per cell)", t_all[rows], c_all[rows])
    unit = "worlds" if draws_per_cell == 1 else f"cells ({draws_per_cell} worlds each)"
    expected_t = expected_c = envelope_t = envelope_c = None
    reference = "no configured law (no sidecar config)"
    if cfg is not None and cfg.active_count_allocation == "stratified":
        targets = allocation_targets(cfg.active_count_weight_matrix(), int(cfg.n_cells))
        targets = np.asarray([[float(v) for v in row] for row in targets])
        expected_t, expected_c = targets.sum(axis=1), targets.sum(axis=0)
        reference = "dots: allocation targets (cells are allocated by design, so no envelope)"
    elif cfg is not None:
        uniform_t = np.full(len(t_axis), 1.0 / len(t_axis))
        uniform_c = np.full(len(c_axis), 1.0 / len(c_axis))
        expected_t, lo_t, hi_t = sidak_envelope(uniform_t, int(rows.size))
        expected_c, lo_c, hi_c = sidak_envelope(uniform_c, int(rows.size))
        reference = "dots: expected under independent uniform counts"
        if draws_per_cell == 1:
            envelope_t, envelope_c = (lo_t, hi_t), (lo_c, hi_c)
            reference += "; whiskers: Šidák 90% envelope"
        else:
            reference += " (no envelope: draws_per_cell > 1)"
    if coverage is not None:
        weight_matrix = np.asarray(coverage["weights"], dtype=np.float64)
    elif cfg is not None:
        weight_matrix = cfg.active_count_weight_matrix()
    else:
        weight_matrix = np.ones(cells.shape)
    excluded = weight_matrix == 0.0
    fig = plt.figure(figsize=(10.0, 8.8), layout="constrained")
    spec = fig.add_gridspec(2, 2, width_ratios=(4.0, 1.25), height_ratios=(1.25, 4.0))
    ax_main = fig.add_subplot(spec[1, 0])
    ax_top = fig.add_subplot(spec[0, 0], sharex=ax_main)
    ax_right = fig.add_subplot(spec[1, 1], sharey=ax_main)
    ax_text = fig.add_subplot(spec[0, 1])
    cmap = plt.colormaps["Blues"].copy()
    cmap.set_bad("#eeeeee")
    ax_main.imshow(
        np.ma.masked_where(excluded, cells),
        cmap=cmap,
        vmin=0,
        vmax=max(1, int(cells.max())),
        origin="lower",
        aspect="auto",
    )
    for i, j in np.ndindex(cells.shape):
        label = "×" if excluded[i, j] else str(cells[i, j])
        shade = "white" if cells[i, j] > 0.6 * max(1, cells.max()) else "black"
        ax_main.text(j, i, label, ha="center", va="center", fontsize=8, color=shade)
    ax_main.set_xticks(range(len(c_axis)), c_axis)
    ax_main.set_yticks(range(len(t_axis)), t_axis)
    ax_main.set_xlabel("active controls (count)")
    ax_main.set_ylabel("active channels (count)")
    ax_main.grid(False)
    _core_marginal(
        ax_top,
        cells.sum(axis=0),
        expected_c,
        envelope_c,
        vertical=True,
        color=_CORE_COLORS["covariate"],
    )
    _core_marginal(
        ax_right,
        cells.sum(axis=1),
        expected_t,
        envelope_t,
        vertical=False,
        color=_CORE_COLORS["treatment"],
    )
    ax_top.set_ylabel(unit, fontsize=8)
    ax_top.set_title("per active-control count", fontsize=8)
    ax_top.tick_params(labelbottom=False)
    ax_right.set_xlabel(unit, fontsize=8)
    ax_right.set_title("per active-channel count", fontsize=8)
    ax_right.tick_params(labelleft=False)
    reached = int((cells > 0).sum())
    possible = int((~excluded).sum())
    ax_text.axis("off")
    excluded_text = (
        f"{int(excluded.sum())} of {excluded.size} have weight 0 (×)\n" if excluded.any() else ""
    )
    ax_text.text(
        0.0,
        0.5,
        f"{report.n_worlds} worlds · {rows.size} cells\n"
        f"{reached} of {possible} weighted combinations reached\n{excluded_text}{rho_text}",
        fontsize=9,
        va="center",
    )
    fig.suptitle(
        f"Active channel × control counts · n {rows.size} cells over "
        f"{len(t_axis)}×{len(c_axis)} combinations",
        fontsize=11,
    )
    plt.show()
    plt.close(fig)
    print(rho_text)
    excluded_caption = (
        "Squares marked × have weight 0 in the allocation, so no world can land there. "
        if excluded.any()
        else ""
    )
    print(
        f"Each square counts the {unit} with that combination of active channels (rows) and "
        f"active controls (columns); the margins add them up per count ({reference}). "
        f"{excluded_caption}Spearman ρ near 0 means the two counts show no monotone "
        "association across the corpus."
    )
    if coverage is not None:
        print("The recount matches the stored active_count_coverage block (n_cells and n_worlds).")

In [ ]:
# ---------------------------------------------------------------------------
# Report figures: trend, zero weeks, shares, example worlds, quality checks and
# exports. Runs after the report core in the same namespace. Private names carry
# a `_fig_` / `_FIG_` prefix so they never shadow a core helper (`_core_`).
# ---------------------------------------------------------------------------

_FIG_ROLES = ("treatment", "covariate")
_FIG_ROLE_NAME = {"treatment": "channels", "covariate": "controls"}
_FIG_SERIES_KEY = {"treatment": "treatment_raw", "covariate": "covariates"}
_FIG_SERIES_UNIT = {"treatment": "raw input units", "covariate": "covariate units"}
_FIG_PREFIX = {"treatment": "C", "covariate": "Z"}
_FIG_COUNT_KEY = {"treatment": "n_treatments_active", "covariate": "n_covariates_active"}
_FIG_PARTS = ("channel", "control", "baseline")
_FIG_COLOR = {"channel": "#306998", "control": "#c0752b", "baseline": "#5b8c5a"}
_FIG_ROLE_COLOR = {"treatment": _FIG_COLOR["channel"], "covariate": _FIG_COLOR["control"]}
_FIG_ALERT = "#b22222"
_FIG_DIRECTIONS = ("down", "flat", "up")
_FIG_TREND_RANGE = {
    "treatment": "treatment_trend_log_change_range",
    "covariate": "covariate_trend_change_range",
}
_FIG_TREND_UNIT = {
    "treatment": "log-level change over the window",
    "covariate": "level change over the window (covariate units)",
}
_FIG_STORAGE_EPS = 2.0**-24  # relative rounding of one value stored as float32
_FIG_SCREEN_VIEW = "differences"
QUALITY_DESCRIPTOR_FEATURES = (
    "treatment_cv_median",
    "treatment_zero_fraction_median",
    "treatment_roughness_median",
    "treatment_acf_lag_1_median",
    "covariate_level_to_variation_median",
    "treatment_pair_diff_abs_pearson_max",
    "outcome_cv",
)
QUALITY_DESCRIPTOR_LAGS = (1, 4, 13, 26, 52)


# ----- shared helpers -------------------------------------------------------------
def _fig_flat_band(report: Report) -> float:
    """The flat band (|change| <= band reads "flat"): the sidecar knob, else the default."""
    band = float((report.knobs or {}).get("FLAT_BAND", DEFAULT_FLAT_BAND))
    if not (math.isfinite(band) and band >= 0.0):
        raise ValueError(f"FLAT_BAND must be a finite number >= 0, got {band!r}")
    return band


def _fig_direction(values: np.ndarray, band: float) -> np.ndarray:
    """Direction codes 0 = down (x < -band), 1 = flat (band edges included), 2 = up."""
    values = np.asarray(values, dtype=np.float64)
    return np.where(values < -band, 0, np.where(values > band, 2, 1))


def _fig_acceptance(report: Report) -> str:
    """The realism-filter acceptance in plain words, for reference-law captions."""
    rate = report.acceptance.get("rejection_rate")
    return "acceptance not recorded" if rate is None else f"acceptance {1.0 - float(rate):.1%}"


def _fig_cell_envelope_note(report: Report) -> str | None:
    """Why a cell-level panel carries no envelope; None when worlds are one per cell."""
    draws = report.corpus["diagnostics"].get("draws_per_cell")
    if draws is None:
        return "no envelope: draws_per_cell not recorded"
    if int(draws) > 1:
        return f"no envelope: draws_per_cell = {int(draws)} (flags repeat within a cell)"
    return None


def _fig_config_range(report: Report, field: str) -> tuple[float, float] | None:
    """A configured (lo, hi) range of the rebuilt ``SCMPrior``; None without a sidecar."""
    cfg = _core_prior(report)
    if cfg is None:
        return None
    lo, hi = (float(value) for value in getattr(cfg, field))
    return lo, hi


def _fig_has_shocks(corpus: dict) -> bool:
    """Whether treatment shocks were drawn (a held shock level overrides the gates)."""
    drawn = np.asarray(corpus["treatment_shock_index"]).shape[1] > 0
    return bool(drawn or np.any(corpus["treatment_shock_mask"]))


def _fig_message(ax, text: str) -> None:
    """Turn an axis into a plain-words note saying why the panel has nothing to draw."""
    ax.set_axis_off()
    ax.text(0.5, 0.5, text, ha="center", va="center", fontsize=9, wrap=True, transform=ax.transAxes)


def _fig_bars(ax, counts, labels, *, color: str, title: str, expected=None):
    """Category bars without an envelope, styled like draw_reference_bars.

    Optional expected-count dots; returns the bar container.
    """
    x = np.arange(len(labels))
    bars = ax.bar(x, counts, width=0.75, color=color, zorder=2)
    if expected is not None:
        ax.scatter(x, expected, s=12, color="black", zorder=4, label="expected count")
        ax.legend(fontsize=7)
    rotate = len(labels) > 6 or max(len(str(label)) for label in labels) > 6
    ax.set_xticks(
        x,
        [str(label) for label in labels],
        rotation=40 if rotate else 0,
        ha="right" if rotate else "center",
    )
    ax.tick_params(labelsize=7)
    ax.set_title(title, fontsize=8)
    return bars


def _fig_spearman(x: np.ndarray, y: np.ndarray) -> str:
    """Spearman's ρ and its two-sided p-value as text; n/a below 3 points or if constant."""
    if x.size < 3:
        return "Spearman ρ n/a (n < 3)"
    if np.ptp(x) == 0 or np.ptp(y) == 0:
        return "Spearman ρ n/a (constant)"
    result = stats.spearmanr(x, y)
    return f"Spearman ρ {result.statistic:+.2f} (p {result.pvalue:.2f})"


# ----- zero weeks -------------------------------------------------------------------
def _fig_gate_on_weeks(report: Report, role: str) -> np.ndarray:
    """On-weeks of every active series, recomputed from the saved gate fields.

    ``prod_g (flag_g == 0 or gate_g(t))`` over g in {onset, offset, flighting}:
    onset is on from its start week, offset before its stop week, flighting while
    ``(t + phase) mod period < on_weeks``. A gate is evaluated only for inputs that
    carry it; a non-carrier's gate fields are zero padding (period 0).
    """
    corpus = report.corpus
    active = active_mask(report, role)
    weeks = np.arange(report.T, dtype=np.int64)[None, :]
    on = np.ones((int(active.sum()), report.T), dtype=bool)
    for component in ("onset", "offset", "flighting"):
        rows = np.flatnonzero(carried_mask(report, role, component)[active])
        if rows.size == 0:
            continue
        prefix = f"trajectory_{role}_{component}_"
        if component == "onset":
            start = np.asarray(corpus[prefix + "start"])[active][rows]
            on[rows] &= weeks >= start[:, None]
        elif component == "offset":
            stop = np.asarray(corpus[prefix + "stop"])[active][rows]
            on[rows] &= weeks < stop[:, None]
        else:
            period, on_weeks, phase = (
                np.asarray(corpus[prefix + leaf])[active][rows]
                for leaf in ("period", "on_weeks", "phase")
            )
            if (period < 2).any():
                raise AssertionError(f"{prefix}period must be >= 2 for every flighting carrier")
            on[rows] &= (weeks + phase[:, None]) % period[:, None] < on_weeks[:, None]
    return on.sum(axis=1)


def zero_counts(report: Report, role: str) -> np.ndarray:
    """Exact zero weeks of every active series (world-major, then slot order), as int64.

    With a trajectory block it asserts the all-gate identity of every series:
    zeros in data == (1 - activity).sum() == T - #{t: prod_g (flag_g == 0 or
    gate_g(t))}, flags from ``*_components`` and gates from the saved onset_start /
    offset_stop / flighting fields. Treatments skip it when shocks exist (a held
    shock level overrides the gates); without the block only observed zeros count.
    """
    if role not in _FIG_ROLES:
        raise ValueError(f"role must be one of {_FIG_ROLES}, got {role!r}")
    corpus = report.corpus
    active = active_mask(report, role)
    series = np.asarray(corpus[_FIG_SERIES_KEY[role]])
    zeros = np.count_nonzero(series == 0, axis=1)[active].astype(np.int64)
    if carried_mask(report, role, "flighting") is None:
        return zeros
    if role == "treatment" and _fig_has_shocks(corpus):
        return zeros
    activity = np.asarray(corpus[f"{role}_activity"], dtype=np.int64)
    off = (report.T - activity.sum(axis=1))[active]
    gated = report.T - _fig_gate_on_weeks(report, role)
    bad = np.flatnonzero((zeros != off) | (off != gated))
    if bad.size:
        world, slot = (int(value) for value in np.argwhere(active)[bad[0]])
        raise AssertionError(
            f"{role} zero identity fails for {bad.size} of {zeros.size} active series; first: "
            f"world {world} slot {slot} has {zeros[bad[0]]} zeros in {_FIG_SERIES_KEY[role]}, "
            f"{off[bad[0]]} off-weeks in {role}_activity and {gated[bad[0]]} from the gates"
        )
    return zeros


def _fig_zero_identity_status(report: Report, role: str) -> str:
    """Which zero identity zero_counts asserted for this role, in plain words."""
    if carried_mask(report, role, "flighting") is None:
        return "observed zeros only (no trajectory block, so the gate identity is not checkable)"
    if role == "treatment" and _fig_has_shocks(report.corpus):
        return "observed zeros only (identity skipped: treatment shocks hold levels)"
    return "zeros in data == (1 − activity) == recomputed gates for every active series"


def _fig_zero_strip(report: Report, zeros: dict[str, np.ndarray], edges: np.ndarray) -> None:
    """One column per world (grouped by its active count), one dot per active series.

    Dots get a small seeded horizontal jitter inside their world's column so series
    with equal zero counts stay visible.
    """
    n = report.n_worlds
    rng = np.random.default_rng(VISUAL_SEED)
    fig, axes = plt.subplots(2, 1, figsize=(14, 7), layout="constrained", squeeze=False)
    for ax, role in zip(axes[:, 0], _FIG_ROLES):
        name = _FIG_ROLE_NAME[role]
        active = active_mask(report, role)
        count = active.sum(axis=1)
        order = np.lexsort((np.arange(n), count))
        column = np.empty(n, dtype=np.int64)
        column[order] = np.arange(n)
        jitter = rng.uniform(-0.3, 0.3, size=zeros[role].size)
        ax.scatter(
            column[np.nonzero(active)[0]] + jitter,
            zeros[role],
            s=10,
            color=_FIG_ROLE_COLOR[role],
            alpha=0.75,
            linewidths=0,
        )
        for edge in edges[1:-1]:
            ax.axhline(edge - 0.5, color="grey", linewidth=0.5, alpha=0.6)
        groups = np.unique(count)
        for group in groups[1:]:
            ax.axvline(column[count == group].min() - 0.5, color="black", linewidth=0.4)
        ax.set_xticks([column[count == group].mean() for group in groups], groups.tolist())
        ax.set(
            xlim=(-0.5, n - 0.5),
            ylim=(-1.5, report.T + 1.5),
            xlabel=f"worlds, grouped by their number of active {name}",
            ylabel="zero weeks",
        )
        ax.set_title(
            f"{name}: zero weeks of every active series, one column per world · "
            f"n {n} worlds, {zeros[role].size} series",
            fontsize=9,
        )
    fig.suptitle("Per-world zero weeks (horizontal lines: category edges)")
    plt.show()
    plt.close(fig)


def plot_zero_weeks(report: Report) -> None:
    """Zero-week categories per role against the exact configured law, plus a per-world strip.

    Categories come from T (0, 1–10, …); every category is drawn, empty ones
    included. The reference is the exact push-forward law of the configured gates
    when the sidecar config allows one (onset = offset = 0, and for treatments no
    shocks); otherwise the bars stand alone with the reason stated.
    """
    edges, labels = zero_category_edges(report.T)
    edges, labels = np.asarray(edges), list(labels)
    zeros = {role: zero_counts(report, role) for role in _FIG_ROLES}
    no_envelope = _fig_cell_envelope_note(report)
    notes = []
    has_law = False
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), layout="constrained", squeeze=False)
    for ax, role in zip(axes[0], _FIG_ROLES):
        name = _FIG_ROLE_NAME[role]
        values = zeros[role]
        category = np.searchsorted(edges, values, side="right") - 1
        if ((category < 0) | (category >= len(labels))).any():
            raise AssertionError(f"{role} zero weeks fall outside the categories {labels}")
        counts = np.bincount(category, minlength=len(labels))
        if int(counts.sum()) != values.size:
            raise AssertionError(f"{role} zero categories do not sum to {values.size} series")
        law = exact_zero_law(report, role)
        title = f"{name}: zero weeks per active series"
        color = _FIG_ROLE_COLOR[role]
        if law is None:
            reason = (
                "configured law unavailable"
                if report.config is None
                else "configured law unavailable: gates other than flighting"
            )
            heading = f"{title} · n {values.size}\n{reason}"
            _fig_bars(ax, counts, labels, color=color, title=heading)
            notes.append(f"{name}: {reason}.")
        else:
            law = np.asarray(law, dtype=np.float64)
            has_law = True
            if law.shape != (len(labels),):
                raise AssertionError(f"exact_zero_law({role!r}) must give one mass per category")
            if no_envelope is None:
                draw_reference_bars(
                    ax, counts, law, int(values.size), labels, color=color, title=title
                )
            else:
                _fig_bars(
                    ax,
                    counts,
                    labels,
                    color=color,
                    title=f"{title} · n {values.size}\n{no_envelope}",
                    expected=values.size * law,
                )
            positive = law[law > 0.0]
            empty = [label for label, mass in zip(labels, law) if mass == 0.0]
            masses = " | ".join(f"{label} {100.0 * mass:.2f}" for label, mass in zip(labels, law))
            notes.append(
                f"{name} exact law (%): {masses}; max/min over positive-mass bins "
                f"{positive.max() / positive.min():.3f}; structurally empty: "
                f"{', '.join(empty) if empty else 'none'}."
            )
        ax.set(xlabel="zero weeks in the reported window", ylabel="active series")
    fig.suptitle(f"Zero weeks per active series (T = {report.T} reported weeks)")
    plt.show()
    plt.close(fig)
    for role in _FIG_ROLES:
        print(f"Zero identity, {_FIG_ROLE_NAME[role]}: {_fig_zero_identity_status(report, role)}.")
    for note in notes:
        print(note)
    caption = (
        "Zero weeks are the exact zeros of each active series over the reported window. "
        "Reference (when available): the exact push-forward of the configured flighting law "
        "(period × duty × phase) mixed with the non-carriers at 0 zeros; markers = expected "
        f"series per category, whiskers = Šidák family-wise {ENVELOPE_LEVEL:.0%} envelope. "
        f"Configured law before the realism filter ({_fig_acceptance(report)})."
    )
    if has_law:
        caption += (
            " With few series the envelopes are wide and flag only gross departures; the "
            "exact law printed above shows how even the configured design itself is."
        )
    print(caption)
    _fig_zero_strip(report, zeros, edges)


# ----- trend ------------------------------------------------------------------------
def _fig_fit_weeks(report: Report, role: str) -> np.ndarray:
    """(world, week, slot) weeks a realized trend is fitted on.

    On-weeks (activity == 1; every week when activity is not recorded); channels also
    drop held shock weeks, where the level can be exactly 0.
    """
    corpus = report.corpus
    activity = corpus.get(f"{role}_activity")
    if activity is None:
        fit = np.ones(np.shape(corpus[_FIG_SERIES_KEY[role]]), dtype=bool)
    else:
        fit = np.asarray(activity) == 1
    if role == "treatment":
        fit &= np.asarray(corpus["treatment_shock_mask"]) == 0
    return fit


def _fig_realized_change(report: Report, role: str) -> np.ndarray:
    """Realized window change of every active series (NaN = n/a: fewer than 2 fit weeks).

    OLS slope over the fit weeks × (T - 1): channels on log level (a log-level
    change), controls on level (covariate units).
    """
    active = active_mask(report, role)
    raw = np.asarray(report.corpus[_FIG_SERIES_KEY[role]], dtype=np.float64)
    series = np.moveaxis(raw, 1, 2)[active]
    fit = np.moveaxis(_fig_fit_weeks(report, role), 1, 2)[active]
    if role == "treatment":
        if (series[fit] <= 0.0).any():
            raise AssertionError("treatment_raw must be positive on every fit week (log level)")
        series = np.log(np.where(fit, series, 1.0))
    n_fit = fit.sum(axis=1)
    change = np.full(n_fit.shape, np.nan)
    ok = n_fit >= 2
    mask, values, count = fit[ok], series[ok], n_fit[ok]
    weeks = np.arange(report.T, dtype=np.float64)
    centred = np.where(mask, weeks - (mask * weeks).sum(axis=1)[:, None] / count[:, None], 0.0)
    level_mean = (mask * values).sum(axis=1) / count
    slope = (centred * (values - level_mean[:, None])).sum(axis=1) / (centred**2).sum(axis=1)
    change[ok] = slope * (report.T - 1)
    return change


def _fig_trend_parameter(report: Report, role: str) -> np.ndarray | None:
    """Recorded trend change of every carried active input; None when not recorded."""
    key = f"trajectory_{role}_trend_change"
    carried = carried_mask(report, role, "trend")
    if carried is None or key not in report.corpus:
        return None
    return np.asarray(report.corpus[key], dtype=np.float64)[carried]


def _fig_trend_parameter_panels(report: Report, role: str, band: float, value_ax, bar_ax) -> None:
    """The drawn trend change of one role by value, and its down/flat/up split."""
    name = _FIG_ROLE_NAME[role]
    color = _FIG_ROLE_COLOR[role]
    drawn = _fig_trend_parameter(report, role)
    if drawn is None or drawn.size == 0:
        if drawn is None:
            note = (
                f"{name}: no trend parameters\n"
                "(no trajectory block, so no input could carry a trend)"
            )
        else:
            note = f"{name}: no active input carries a trend"
        _fig_message(value_ax, note)
        _fig_message(bar_ax, note)
        return
    law = _fig_config_range(report, _FIG_TREND_RANGE[role])
    low, high = (float(drawn.min()), float(drawn.max())) if law is None else law
    low, high = min(low, float(drawn.min())), max(high, float(drawn.max()))
    half = band if band > 0.0 else 0.5  # one visible bin around a point mass
    edges = np.linspace(low, high, 13) if high > low else np.array([low - half, low + half])
    counts = np.histogram(drawn, bins=edges)[0]
    edgecolors = ["white"] * counts.size
    support = "configured law unavailable"
    if law is not None and law[1] > law[0]:
        overlap = np.minimum(edges[1:], law[1]) - np.maximum(edges[:-1], law[0])
        probs = np.clip(overlap, 0.0, None) / (law[1] - law[0])
        expected, lower, upper = sidak_envelope(probs, int(drawn.size))
        outside = (counts < lower) | (counts > upper)
        edgecolors = ["#c0392b" if out else "white" for out in outside]
        _core_envelope_marks(value_ax, 0.5 * (edges[:-1] + edges[1:]), expected, lower, upper)
        support = (
            f"configured U({law[0]:g}, {law[1]:g}) · {int(outside.sum())} of {counts.size} "
            "bins outside"
        )
    elif law is not None:
        support = f"configured point mass {law[0]:g}"
    value_ax.bar(
        edges[:-1],
        counts,
        width=np.diff(edges),
        align="edge",
        color=color,
        alpha=0.85,
        edgecolor=edgecolors,
        linewidth=1.0,
        zorder=2,
    )
    value_ax.axvspan(-band, band, color="grey", alpha=0.15, label=f"flat band ±{band:.3g}")
    value_ax.set(xlabel=_FIG_TREND_UNIT[role], ylabel="carried active inputs")
    value_ax.set_title(
        f"{name}: drawn trend change by value · n {drawn.size}\n{support}", fontsize=9
    )
    value_ax.legend(fontsize=7)
    counts = np.bincount(_fig_direction(drawn, band), minlength=3)
    if law is None:
        title = f"{name}: drawn direction · n {drawn.size}\nconfigured law unavailable"
        _fig_bars(bar_ax, counts, _FIG_DIRECTIONS, color=color, title=title)
    else:
        probs = np.asarray(trend_category_law(law[0], law[1], band), dtype=np.float64)
        title = f"{name}: drawn direction vs U({law[0]:g}, {law[1]:g}) given carrying"
        draw_reference_bars(
            bar_ax, counts, probs, int(drawn.size), list(_FIG_DIRECTIONS), color=color, title=title
        )
    bar_ax.set_ylabel("carried active inputs")


def _fig_trend_realized_panels(
    report: Report, role: str, band: float, change: np.ndarray, bar_ax, scatter_ax
) -> None:
    """Realized direction bars of one role and, with scatter_ax, parameter vs realized change."""
    name = _FIG_ROLE_NAME[role]
    color = _FIG_ROLE_COLOR[role]
    unit = _FIG_TREND_UNIT[role]
    fitted = np.isfinite(change)
    counts = np.bincount(_fig_direction(change[fitted], band), minlength=3)
    n_na = int((~fitted).sum())
    title = f"{name}: realized direction · n {int(fitted.sum())} series ({n_na} n/a)"
    bars = _fig_bars(bar_ax, counts, _FIG_DIRECTIONS, color=color, title=title)
    total = max(1, int(counts.sum()))
    bar_ax.bar_label(bars, labels=[f"{c} ({c / total:.0%})" for c in counts.tolist()], fontsize=8)
    bar_ax.set_ylabel("active series")
    if scatter_ax is None:
        return
    drawn = _fig_trend_parameter(report, role)
    if drawn is None:
        _fig_message(scatter_ax, f"{name}: trend parameter not recorded (realized directions only)")
        return
    carriers = carried_mask(report, role, "trend")[active_mask(report, role)]
    if not carriers.any():
        _fig_message(scatter_ax, f"{name}: no active input carries a trend")
        return
    observed = change[carriers]
    fit = np.isfinite(observed)
    if not fit.any():
        _fig_message(scatter_ax, f"{name}: no trend carrier with 2 or more fit weeks")
        return
    x, y = drawn[fit], observed[fit]
    same = _fig_direction(x, band) == _fig_direction(y, band)
    scatter_ax.scatter(x[same], y[same], s=14, color=color, label=f"same direction ({same.sum()})")
    scatter_ax.scatter(
        x[~same], y[~same], s=18, marker="x", color=_FIG_ALERT, label=f"other ({(~same).sum()})"
    )
    for edge in (-band, band):
        scatter_ax.axhline(edge, color="grey", linewidth=0.6, linestyle=":")
        scatter_ax.axvline(edge, color="grey", linewidth=0.6, linestyle=":")
    span = [float(x.min()), float(x.max())]
    scatter_ax.plot(span, span, color="black", linewidth=0.8, label="realized = parameter")
    scatter_ax.set(xlabel=f"trend parameter ({unit})", ylabel=f"realized ({unit})")
    scatter_ax.set_title(
        f"{name}: parameter vs realized · n {x.size} trend carriers ({int((~fit).sum())} n/a)\n"
        f"same direction {int(same.sum())} of {x.size} ({same.mean():.0%})",
        fontsize=9,
    )
    scatter_ax.legend(fontsize=7)


def plot_trend(report: Report) -> None:
    """Trend parameter view (what was drawn) and realized view (what the series did)."""
    band = _fig_flat_band(report)
    drawn = {role: _fig_trend_parameter(report, role) for role in _FIG_ROLES}
    block = any(values is not None for values in drawn.values())
    recorded = any(values is not None and values.size > 0 for values in drawn.values())
    if recorded:
        fig, axes = plt.subplots(2, 2, figsize=(13, 8), layout="constrained", squeeze=False)
        for column, role in enumerate(_FIG_ROLES):
            _fig_trend_parameter_panels(report, role, band, axes[0, column], axes[1, column])
        fig.suptitle(
            f"Trend parameter — what the generator drew (flat = |change| ≤ {band:.3g}; "
            "channels: log-level change, controls: level change)"
        )
        plt.show()
        plt.close(fig)
        if report.config is None:
            reference = "Configured law unavailable (no sidecar config): histograms only."
        else:
            reference = (
                "Category reference: the configured trend law given carrying, U(lo, hi) split "
                f"at ±{band:.3g}; markers = expected count, whiskers = Šidák family-wise "
                f"{ENVELOPE_LEVEL:.0%} envelope. Configured law before the realism filter "
                f"({_fig_acceptance(report)})."
            )
        print(
            "Parameter view: the drawn trend change of every active input that carries a "
            f"trend; how often inputs carry one is shown in the parameter grid. {reference}"
        )
    elif block:
        print(
            "Trend parameter view: no active input carries a trend (trend inclusion "
            "probability 0 or no carrier drawn); the realized directions below need only the "
            "series."
        )
    else:
        print(
            "Trend parameter view: no trend parameters (no trajectory block, so no input "
            "could carry a trend); the realized directions below need only the series."
        )
    changes = {role: _fig_realized_change(report, role) for role in _FIG_ROLES}
    rows = 2 if recorded else 1
    fig, axes = plt.subplots(rows, 2, figsize=(13, 4.4 * rows), layout="constrained", squeeze=False)
    for column, role in enumerate(_FIG_ROLES):
        scatter_ax = axes[1, column] if recorded else None
        _fig_trend_realized_panels(report, role, band, changes[role], axes[0, column], scatter_ax)
    fig.suptitle(
        "Realized direction — what each active series did over the reported window "
        f"(OLS slope on fit weeks × (T − 1); flat = |change| ≤ {band:.3g})"
    )
    plt.show()
    plt.close(fig)
    n_na = {role: int(np.isnan(changes[role]).sum()) for role in _FIG_ROLES}
    print(
        "Realized view: window change = OLS slope over the fit weeks × (T − 1); channels on "
        "log level (log-level change), controls on level (covariate units). Fit weeks are "
        "on-weeks (activity == 1; every week when activity is not recorded), channels also "
        "without held shock weeks, so off-weeks never fake a trend. n/a (fewer than 2 fit "
        f"weeks, left out of the bars): channels {n_na['treatment']}, controls "
        f"{n_na['covariate']}. The remaining gap between parameter and realized direction is "
        "each series' own walk drift, seasonality and level jumps; narrower walk-std ranges "
        "in the knobs would tighten it, a modelling choice left to the user."
    )


# ----- shares -----------------------------------------------------------------------
def _fig_beta12_cdf(x: np.ndarray) -> np.ndarray:
    """CDF of Beta(1, 2): one share's marginal when the 3-part composition is uniform."""
    u = np.clip(np.asarray(x, dtype=np.float64), 0.0, 1.0)
    return 1.0 - (1.0 - u) ** 2


def _fig_beta12_extremes(n: int, q) -> tuple[np.ndarray, np.ndarray]:
    """Quantiles ``q`` of the min and of the max of n iid Beta(1, 2) shares."""
    q = np.asarray(q, dtype=np.float64)
    minimum = 1.0 - (1.0 - q) ** (1.0 / (2 * n))
    maximum = 1.0 - np.sqrt(1.0 - q ** (1.0 / n))
    return minimum, maximum


def _fig_share_histogram(ax, values: np.ndarray, part: str, n: int) -> None:
    """One part's share histogram against the uniform-composition ideal.

    Bins outside the ideal's envelope are outlined in red. The realized min and max are
    checked against the ideal's range at n, Šidák-adjusted over the six share extremes
    (min and max of three parts): with few worlds every lower whisker is 0, so only this
    range check can show shares that never occur.
    """
    low = math.floor(10.0 * min(0.0, float(values.min())))
    high = math.ceil(10.0 * max(1.0, float(values.max())))
    edges = np.arange(low, high + 1) / 10.0
    counts = np.histogram(values, bins=edges)[0]
    probs = np.diff(_fig_beta12_cdf(edges))
    expected, lower, upper = sidak_envelope(probs, n)
    flagged = (counts < lower) | (counts > upper)
    outside = int(flagged.sum())
    centres = 0.5 * (edges[:-1] + edges[1:])
    ax.bar(
        edges[:-1],
        counts,
        width=np.diff(edges),
        align="edge",
        color=_FIG_COLOR[part],
        alpha=0.85,
        edgecolor=["#c0392b" if out else "white" for out in flagged],
        linewidth=1.0,
        label="worlds",
    )
    ax.vlines(
        centres,
        lower,
        upper,
        color="black",
        linewidth=1.2,
        label=f"ideal's Šidák family-wise {ENVELOPE_LEVEL:.0%} envelope",
    )
    ax.plot(
        centres,
        expected,
        linestyle="none",
        marker="o",
        markersize=3.5,
        color="black",
        label="uniform composition (Beta(1,2) marginal) — ideal, not the generator’s law",
    )
    ax.set_xlim(edges[0], edges[-1])
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    alpha = 1.0 - ENVELOPE_LEVEL ** (1.0 / 6.0)  # Šidák over the six share extremes
    minimum, maximum = _fig_beta12_extremes(n, [alpha / 2.0, 0.5, 1.0 - alpha / 2.0])
    low_out = not minimum[0] <= float(values.min()) <= minimum[2]
    high_out = not maximum[0] <= float(values.max()) <= maximum[2]
    ax.text(
        0.98,
        0.97,
        f"realized min {values.min():.1%}{' (outside)' if low_out else ''} · "
        f"max {values.max():.1%}{' (outside)' if high_out else ''}\n"
        f"ideal at n {n}: median [family-wise {ENVELOPE_LEVEL:.0%} range]\n"
        f"min {minimum[1]:.1%} [{minimum[0]:.1%}, {minimum[2]:.1%}] · "
        f"max {maximum[1]:.1%} [{maximum[0]:.1%}, {maximum[2]:.1%}]",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=7,
        bbox={
            "boxstyle": "round",
            "facecolor": "white",
            "edgecolor": "#c0392b" if low_out or high_out else "black",
            "alpha": 0.85,
        },
    )
    extremes = [word for word, out in (("min", low_out), ("max", high_out)) if out]
    range_note = f" · {' and '.join(extremes)} outside the ideal's range" if extremes else ""
    ax.set(xlabel="share of the reported-window outcome", ylabel="worlds")
    ax.set_title(
        f"{part} share · n {n} worlds\n{outside} of {counts.size} bins outside{range_note}",
        fontsize=9,
    )


def _fig_ternary_xy(control: np.ndarray, baseline: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Exact barycentric position: channel corner (0, 0), control (1, 0), baseline on top."""
    control, baseline = np.asarray(control), np.asarray(baseline)
    return control + 0.5 * baseline, (math.sqrt(3.0) / 2.0) * baseline


def _fig_count(n: int, noun: str) -> str:
    """``n`` with the singular or plural ``noun``."""
    return f"{n} {noun}" + ("" if n == 1 else "s")


def _fig_composition_triangle(shares: dict[str, np.ndarray], n: int) -> None:
    """Every world's composition in the triangle; worlds with a negative part marked."""
    x, y = _fig_ternary_xy(shares["control"], shares["baseline"])
    negative = (shares["channel"] < 0) | (shares["control"] < 0) | (shares["baseline"] < 0)
    top = math.sqrt(3.0) / 2.0
    fig, ax = plt.subplots(figsize=(7, 6.8), layout="constrained")
    for level in (0.2, 0.4, 0.6, 0.8):
        rest = 1.0 - level
        # (control, baseline) endpoints of constant channel, control and baseline share
        for (c0, b0), (c1, b1) in (
            ((rest, 0.0), (0.0, rest)),
            ((level, 0.0), (level, rest)),
            ((0.0, level), (rest, level)),
        ):
            gx, gy = _fig_ternary_xy(np.array([c0, c1]), np.array([b0, b1]))
            ax.plot(gx, gy, color="grey", linewidth=0.5, alpha=0.5)
    ax.plot([0.0, 1.0, 0.5, 0.0], [0.0, 0.0, top, 0.0], color="black", linewidth=1.0)
    ax.text(0.0, -0.04, "100% channel", ha="center", va="top", fontsize=9)
    ax.text(1.0, -0.04, "100% control", ha="center", va="top", fontsize=9)
    ax.text(0.5, top + 0.03, "100% baseline", ha="center", va="bottom", fontsize=9)
    ax.scatter(
        x[~negative],
        y[~negative],
        s=18,
        color=_FIG_COLOR["channel"],
        alpha=0.8,
        label=_fig_count(int((~negative).sum()), "world") + " with three non-negative parts",
    )
    ax.scatter(
        x[negative],
        y[negative],
        s=40,
        marker="x",
        color=_FIG_ALERT,
        label=_fig_count(int(negative.sum()), "world") + " with a negative part (off the triangle)",
    )
    ax.set(
        xlim=(min(-0.12, float(x.min()) - 0.05), max(1.12, float(x.max()) + 0.05)),
        ylim=(min(-0.12, float(y.min()) - 0.05), max(top + 0.1, float(y.max()) + 0.05)),
    )
    ax.set_aspect("equal")
    ax.set_axis_off()
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=1, fontsize=8)
    ax.set_title(
        f"Composition of every world · n {n} worlds (grid lines every 20% of a part)", fontsize=10
    )
    plt.show()
    plt.close(fig)


def _fig_mean_active_fraction(report: Report, role: str) -> tuple[np.ndarray, bool]:
    """Per world, the on-week fraction averaged over its active inputs (≡ 1 without activity)."""
    active = active_mask(report, role)
    activity = report.corpus.get(f"{role}_activity")
    recorded = activity is not None
    on = np.asarray(activity, dtype=np.float64).mean(axis=1) if recorded else np.ones(active.shape)
    count = active.sum(axis=1)
    fraction = np.full(report.n_worlds, np.nan)
    has = count > 0
    fraction[has] = (on * active).sum(axis=1)[has] / count[has]
    return fraction, recorded


def _fig_driver_scatter(ax, x, share: np.ndarray, *, part: str, xlabel: str) -> None:
    """One realized driver against one share, with Spearman's rank correlation."""
    x = np.asarray(x, dtype=np.float64)
    keep = np.isfinite(x)
    ax.scatter(x[keep], share[keep], s=16, color=_FIG_COLOR[part], alpha=0.8)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.set_xlabel(xlabel, fontsize=8)
    ax.set_ylabel(f"{part} share")
    ax.set_title(
        f"{part} share · n {int(keep.sum())} worlds · {_fig_spearman(x[keep], share[keep])}",
        fontsize=9,
    )


def _fig_share_drivers(report: Report, shares: dict[str, np.ndarray]) -> None:
    """Each share against the realized quantities that move it, one dot per world."""
    corpus = report.corpus
    direct = np.asarray(corpus["treatment_contribution_raw"], dtype=np.float64).sum(axis=(1, 2))
    indirect = np.asarray(corpus["indirect_effects"], dtype=np.float64).sum(axis=1)
    control = np.asarray(corpus["covariate_contribution"], dtype=np.float64).sum(axis=(1, 2))
    totals = {"channel": direct + indirect, "control": control}
    fig, axes = plt.subplots(3, 3, figsize=(14, 11), layout="constrained", squeeze=False)
    for row, (part, role) in enumerate((("channel", "treatment"), ("control", "covariate"))):
        name = _FIG_ROLE_NAME[role]
        count = active_mask(report, role).sum(axis=1)
        fraction, recorded = _fig_mean_active_fraction(report, role)
        per_input = np.full(report.n_worlds, np.nan)
        has = count > 0
        per_input[has] = totals[part][has] / (report.T * count[has])
        fraction_label = f"mean active fraction of the active {name}"
        if not recorded:
            fraction_label += " (activity not recorded: ≡ 1)"
        drivers = (
            (count, f"active {name} (count)"),
            (fraction, fraction_label),
            (per_input, f"mean {part} contribution per active input-week (outcome units)"),
        )
        for ax, (x, label) in zip(axes[row], drivers):
            _fig_driver_scatter(ax, x, shares[part], part=part, xlabel=label)
    inputs = sum(active_mask(report, role).sum(axis=1) for role in _FIG_ROLES)
    intrinsic = np.asarray(corpus["baseline_intrinsic"], dtype=np.float64).mean(axis=1)
    baseline_drivers = (
        (inputs, "active channels + controls (count)"),
        (intrinsic, "realized intrinsic baseline mean (outcome units)"),
    )
    for ax, (x, label) in zip(axes[2], baseline_drivers):
        _fig_driver_scatter(ax, x, shares["baseline"], part="baseline", xlabel=label)
    axes[2, 2].set_axis_off()
    fig.suptitle(f"What moves each share: realized quantities per world · n {report.n_worlds}")
    plt.show()
    plt.close(fig)


def plot_shares(report: Report) -> None:
    """Share histograms against the uniform-composition ideal, the triangle and the drivers."""
    shares = realized_shares(report, show_closure=True)
    shares = {part: np.asarray(shares[part], dtype=np.float64) for part in _FIG_PARTS}
    n = report.n_worlds
    fig, axes = plt.subplots(
        1, 3, figsize=(15, 5), layout="constrained", sharey=True, squeeze=False
    )
    for ax, part in zip(axes[0], _FIG_PARTS):
        _fig_share_histogram(ax, shares[part], part, n)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=3, fontsize=8)
    fig.suptitle("Channel, control and baseline shares of each world's reported-window outcome")
    plt.show()
    plt.close(fig)
    print(
        "Shares of each world's reported-window outcome: channel = Σ direct treatment "
        "contributions + Σ indirect effects; control = Σ covariate contributions (net: "
        "covariate series are signed, so it can be negative); baseline = Σ intrinsic baseline "
        "+ Σ latent contributions + Σ observation noise (noise folded in). The three sum to "
        "100% per world, so they cannot all be flat on 0–100%: a composition drawn uniformly "
        "over the triangle gives each share the Beta(1, 2) marginal, density 2(1 − x). That "
        "ideal, not the generator's law, is the reference: markers = expected worlds per "
        f"bin, whiskers = its Šidák family-wise {ENVELOPE_LEVEL:.0%} envelope at n {n}, and "
        "red-outlined bins leave it. With few worlds every lower whisker is 0, so the bins "
        "flag only crowding, never missing shares; the box is the range check: the realized "
        "min and max against the ideal's range at n (Šidák-adjusted over the six extremes), "
        "marked 'outside' when they leave it."
    )
    _fig_composition_triangle(shares, n)
    print(
        "Each world sits at its exact barycentric position (a corner = 100% of one part); "
        "compositions spread evenly over the triangle are the uniform ideal above. Worlds "
        "with a negative part (a net-negative control effect) fall outside and are marked ×."
    )
    _fig_share_drivers(report, shares)
    print(
        "Each dot is one world and every axis a realized quantity (no fitted model); ρ is "
        "Spearman's rank correlation and p its two-sided p-value: with n worlds, |ρ| below "
        f"about 2/√n ({2.0 / math.sqrt(n):.2f} here) is within chance. Mean active fraction "
        "= on-weeks / T averaged over the world's active inputs."
    )


# ----- example worlds ---------------------------------------------------------------
def pick_example_worlds(report: Report, seed: int = VISUAL_SEED) -> list[tuple[int, list[str]]]:
    """Example worlds: the share extremes, then one seeded pick from the other worlds.

    The lowest and highest world of each share, in channel / control / baseline
    order; a world extreme twice is listed once with both labels. One world drawn
    with ``seed`` from the remaining worlds follows, when any remain.
    """
    shares = realized_shares(report)
    picks: dict[int, list[str]] = {}
    for part in _FIG_PARTS:
        values = np.asarray(shares[part], dtype=np.float64)
        for word, world in (("lowest", np.argmin(values)), ("highest", np.argmax(values))):
            picks.setdefault(int(world), []).append(f"{word} {part} share")
    remaining = [world for world in range(report.n_worlds) if world not in picks]
    if remaining:
        rng = np.random.default_rng(seed)
        picks[int(rng.choice(remaining))] = [f"seeded pick from the other worlds (seed {seed})"]
    return list(picks.items())


def _fig_world_decomposition(
    corpus: dict, world: int
) -> tuple[dict[str, np.ndarray], np.ndarray, float]:
    """Weekly channel / control / baseline parts of one world, closure asserted per week.

    Each stored float32 value carries at most 2^-24 relative rounding, so
    |Σ parts − outcome| <= 2^-24 · (|outcome| + Σ |stored part|) every week.
    """
    direct = np.asarray(corpus["treatment_contribution_raw"][world], dtype=np.float64)
    indirect = np.asarray(corpus["indirect_effects"][world], dtype=np.float64)
    control = np.asarray(corpus["covariate_contribution"][world], dtype=np.float64)
    intrinsic = np.asarray(corpus["baseline_intrinsic"][world], dtype=np.float64)
    latent = np.asarray(corpus["latent_unobserved_contribution"][world], dtype=np.float64)
    noise = np.asarray(corpus["outcome_noise"][world], dtype=np.float64)
    outcome = np.asarray(corpus["outcome_raw"][world], dtype=np.float64)
    parts = {
        "channel": direct.sum(axis=1) + indirect,
        "control": control.sum(axis=1),
        "baseline": intrinsic + latent.sum(axis=1) + noise,
    }
    residual = np.abs(parts["channel"] + parts["control"] + parts["baseline"] - outcome)
    magnitude = (
        np.abs(outcome)
        + np.abs(direct).sum(axis=1)
        + np.abs(indirect)
        + np.abs(control).sum(axis=1)
        + np.abs(intrinsic)
        + np.abs(latent).sum(axis=1)
        + np.abs(noise)
    )
    excess = residual - _FIG_STORAGE_EPS * magnitude
    if (excess > 0.0).any():
        week = int(np.argmax(excess))
        raise AssertionError(
            f"world {world} week {week}: channel + control + baseline misses outcome_raw by "
            f"{residual[week]:.3g}, above the float32 storage bound "
            f"{_FIG_STORAGE_EPS * magnitude[week]:.3g}"
        )
    return parts, outcome, float(residual.max())


def plot_world(report: Report, world_id: int, labels: list[str]) -> None:
    """One world: its active inputs and the exact three-part outcome decomposition."""
    corpus = report.corpus
    world = int(world_id)
    if not 0 <= world < report.n_worlds:
        raise ValueError(f"world_id must be in [0, {report.n_worlds}), got {world_id!r}")
    parts, outcome, residual = _fig_world_decomposition(corpus, world)
    shares = realized_shares(report)
    weeks = np.arange(report.T)
    fig, axes = plt.subplots(3, 1, figsize=(12, 9.5), sharex=True, layout="constrained")
    for ax, role in zip(axes[:2], _FIG_ROLES):
        slots = np.flatnonzero(active_mask(report, role)[world])
        series = np.asarray(corpus[_FIG_SERIES_KEY[role]][world])
        for slot in slots:
            ax.plot(weeks, series[:, slot], linewidth=1, label=f"{_FIG_PREFIX[role]}{slot + 1}")
        ax.set_ylabel(_FIG_SERIES_UNIT[role])
        ax.set_title(
            f"{_FIG_ROLE_NAME[role]}: {_fig_count(int(slots.size), 'active input')} "
            f"({_FIG_SERIES_UNIT[role]})",
            fontsize=9,
        )
        if slots.size:
            ncol = 1 if slots.size <= 8 else 2
            ax.legend(ncol=ncol, fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
    ax = axes[2]
    ax.plot(weeks, outcome, color="black", linewidth=1.6, label="outcome")
    for part in _FIG_PARTS:
        ax.plot(
            weeks,
            parts[part],
            color=_FIG_COLOR[part],
            linewidth=1.1,
            label=f"{part} ({float(shares[part][world]):.1%} of the window's outcome)",
        )
    ax.axhline(0.0, color="grey", linewidth=0.6)
    ax.set(xlabel="reported week (zero-based)", ylabel="outcome units")
    ax.set_title(
        "exact decomposition: max |channel + control + baseline − outcome| = "
        f"{residual:.2g} outcome units (float32 storage rounding)",
        fontsize=9,
    )
    ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))
    fig.suptitle(
        f"World {world} — {'; '.join(labels)} · cell {int(corpus['cell_id'][world])} · "
        f"T = {report.T} weeks"
    )
    plt.show()
    plt.close(fig)


# ----- #31 quality checks -----------------------------------------------------------
def _fig_quality_prevalence(report: Report) -> None:
    """Carried-component counts vs the configured probability, recomputed == diagnostics."""
    corpus = report.corpus
    block = corpus["diagnostics"].get("trajectory")
    if block is None:
        print("Prevalence vs configured probability: not recorded (no trajectory block).")
        return
    if tuple(block["components"]) != TRAJECTORY_COMPONENTS:
        raise AssertionError("diagnostics trajectory components differ from TRAJECTORY_COMPONENTS")
    no_envelope = _fig_cell_envelope_note(report)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), layout="constrained", squeeze=False)
    for ax, role in zip(axes[0], _FIG_ROLES):
        active = active_mask(report, role)
        n = int(active.sum())
        counts = np.asarray(corpus[f"{role}_components"])[active].sum(axis=0).astype(np.int64)
        if int(block["n_inputs"][role]) != n:
            raise AssertionError(
                f"diagnostics trajectory n_inputs[{role!r}] differs from the masks"
            )
        for component, count in zip(TRAJECTORY_COMPONENTS, counts.tolist()):
            share = float(count) / n if n else 0.0
            if block["prevalence"][role][component] != share:
                raise AssertionError(
                    f"diagnostics trajectory prevalence[{role!r}][{component!r}] differs from "
                    "the mask recomputation"
                )
        probs = np.array([float(block["inclusion_probs"][role][c]) for c in TRAJECTORY_COMPONENTS])
        title = f"{_FIG_ROLE_NAME[role]}: carried components vs configured probability"
        color = _FIG_ROLE_COLOR[role]
        if no_envelope is None:
            draw_reference_bars(
                ax, counts, probs, n, list(TRAJECTORY_COMPONENTS), color=color, title=title
            )
        else:
            _fig_bars(
                ax,
                counts,
                list(TRAJECTORY_COMPONENTS),
                color=color,
                title=f"{title} · n {n}\n{no_envelope}",
                expected=n * probs,
            )
        ax.set_ylabel("active inputs carrying the component")
    fig.suptitle("Component prevalence (recomputed from the masks == stored diagnostics)")
    plt.show()
    plt.close(fig)
    envelope = (
        f", whiskers its Šidák family-wise {ENVELOPE_LEVEL:.0%} envelope over the eight components"
        if no_envelope is None
        else f" ({no_envelope})"
    )
    print(
        "Bars count the active inputs carrying each component; markers are the expected count "
        "n · p for the effective inclusion probability p stored in the corpus diagnostics"
        f"{envelope}. Configured probability before the realism filter "
        f"({_fig_acceptance(report)})."
    )


def _fig_quality_coverage(report: Report) -> None:
    """The configured active-count grid against the realized worlds per combination."""
    corpus = report.corpus
    block = corpus["diagnostics"].get("active_count_coverage")
    if block is None:
        print("Grid vs coverage: not recorded (no active_count_coverage block).")
        return
    coverage = pg.active_count_coverage(corpus)
    for key in ("n_treatments_active", "n_covariates_active", "n_cells", "n_worlds"):
        if coverage[key] != block[key]:
            raise AssertionError(f"diagnostics active_count_coverage {key} differs from recount")
    positive = np.asarray(block["weights"], dtype=np.float64) > 0.0
    worlds = np.asarray(coverage["n_worlds"], dtype=np.int64)
    cells = np.asarray(coverage["n_cells"], dtype=np.int64)
    if (worlds[~positive] > 0).any():
        raise AssertionError("worlds were allocated to zero-weight count combinations")
    reached = int((cells[positive] > 0).sum())
    print(
        f"Grid coverage: the recount from the masks equals the stored diagnostics; {reached} of "
        f"{int(positive.sum())} positive-weight (channels, controls) combinations reached and no "
        "world sits on a zero-weight combination (the grid itself is drawn in section 1)."
    )


def _fig_design_screen(report: Report) -> dict[str, np.ndarray]:
    """Per-world identifiability screen on the differenced observed design.

    The design_screen rule of the earlier notebook: a world is assessed when it has a
    predictor, at least 3 steps and a valid VIF for every varying predictor; it is
    flagged when its design is rank-deficient or any predictor's VIF >= VIF_THRESHOLD.
    """
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore", RuntimeWarning)
        diagnostics = pg.data_diagnostics(
            report.corpus,
            scopes=("nodes",),
            views=(_FIG_SCREEN_VIEW,),
            lags=(1,),
            keep_series=False,
        )
    result = diagnostics[_FIG_SCREEN_VIEW].vif["observed"]
    if not np.array_equal(result.world_ids, np.arange(report.n_worlds)):
        raise AssertionError("the identifiability screen must cover every world in order")
    n = result.n_worlds
    screen = {
        "predictors": np.zeros(n, dtype=np.int64),
        "max_vif": np.full(n, np.nan),
        "assessed": np.zeros(n, dtype=bool),
        "deficient": np.zeros(n, dtype=bool),
        "collinear": np.zeros(n, dtype=bool),
    }
    for position in range(n):
        eligible = result.eligible[position]
        valid = result.valid[position]
        values = result.vif[position, valid]
        varying = eligible & ~result.constant[position]
        n_predictors = int(eligible.sum())
        screen["predictors"][position] = n_predictors
        screen["assessed"][position] = (
            n_predictors > 0 and result.n_time_steps >= 3 and bool(valid[varying].all())
        )
        if values.size:
            screen["max_vif"][position] = float(values.max())
            screen["collinear"][position] = bool((values >= VIF_THRESHOLD).any())
        screen["deficient"][position] = int(result.rank[position]) < n_predictors
    return screen


def _fig_quality_identifiability(report: Report) -> None:
    """The documented identifiability criterion as a plot, with its PASS/FAIL decision."""
    screen = _fig_design_screen(report)
    n = report.n_worlds
    flagged = screen["deficient"] | screen["collinear"]
    assessed = screen["assessed"]
    n_unassessed = int((~assessed).sum())
    share = float(flagged.mean())
    status = "UNASSESSED" if n_unassessed else ("PASS" if share <= MAX_BAD_SHARE else "FAIL")
    rng = np.random.default_rng(VISUAL_SEED)
    x = screen["predictors"] + rng.uniform(-0.2, 0.2, size=n)
    y = screen["max_vif"].copy()
    infinite = np.isposinf(y)
    finite = np.isfinite(y)
    top = 10.0 * VIF_THRESHOLD
    if finite.any():
        top = max(top, 3.0 * float(y[finite].max()))
    y[infinite] = top
    y[np.isnan(y)] = 1.0
    fig, ax = plt.subplots(figsize=(11, 4.8), layout="constrained")
    ax.axhline(
        VIF_THRESHOLD,
        color=_FIG_ALERT,
        linestyle="--",
        linewidth=1,
        label=f"VIF = {VIF_THRESHOLD:g} threshold",
    )
    flagged_label = f"flagged: rank-deficient or VIF ≥ {VIF_THRESHOLD:g}"
    groups = (
        (assessed & ~flagged, "o", _FIG_COLOR["channel"], "not flagged"),
        (assessed & flagged & ~infinite, "o", _FIG_ALERT, flagged_label),
        (assessed & infinite, "^", _FIG_ALERT, "flagged: infinite VIF, drawn at the top"),
        (~assessed, "s", "grey", "unassessed (drawn at 1 without a valid VIF)"),
    )
    for selected, marker, color, label in groups:
        if selected.any():
            ax.scatter(x[selected], y[selected], s=22, marker=marker, color=color, label=label)
    ax.set_yscale("log")
    ax.set(
        xlabel="active predictors in the observed design (channels + controls)",
        ylabel="largest VIF among the predictors (unitless)",
    )
    ax.set_title(
        f"Identifiability screen · {_FIG_SCREEN_VIEW} view · n {n} worlds: "
        f"{int(flagged.sum())} flagged ({share:.0%}) → {status} (max bad share "
        f"{MAX_BAD_SHARE:.0%})",
        fontsize=10,
    )
    ax.legend(fontsize=7, loc="upper left")
    plt.show()
    plt.close(fig)
    print(
        f"Identifiability: {status} — {int(flagged.sum())} of {n} worlds flagged ({share:.0%}; "
        f"rank-deficient {int(screen['deficient'].sum())}, VIF ≥ {VIF_THRESHOLD:g} "
        f"{int(screen['collinear'].sum())}), {n_unassessed} unassessed."
    )
    print(
        f"Criterion: the {_FIG_SCREEN_VIEW} (first differences) of each world's active channels "
        "and controls form its observed design; a world is flagged when that design is "
        f"rank-deficient or any predictor's VIF is ≥ {VIF_THRESHOLD:g}. The corpus passes when "
        f"at most {MAX_BAD_SHARE:.0%} of worlds are flagged, and is UNASSESSED when any world "
        "lacks a predictor, has fewer than 3 steps, or has a varying predictor without a "
        "valid VIF."
    )


def _fig_quality_descriptors(report: Report) -> None:
    """Histograms of the scale-free observable world descriptors."""
    with warnings.catch_warnings(), np.errstate(all="ignore"):
        warnings.simplefilter("ignore", RuntimeWarning)
        rows = pg.world_descriptors(report.corpus, source_id="report", lags=QUALITY_DESCRIPTOR_LAGS)
    fig, axes = plt.subplots(2, 4, figsize=(14, 6.4), layout="constrained", squeeze=False)
    for ax, feature in zip(axes.flat, QUALITY_DESCRIPTOR_FEATURES):
        values = rows.column(feature)
        finite = np.isfinite(values)
        if finite.any():
            bins = min(10, max(3, int(math.sqrt(int(finite.sum())))))
            ax.hist(values[finite], bins=bins, color=_FIG_COLOR["channel"], edgecolor="white")
            ax.set_ylabel("worlds")
        else:
            _fig_message(ax, "no finite value")
        ax.set_title(
            f"{feature} (unitless)\nfinite {int(finite.sum())} of {values.size} worlds", fontsize=8
        )
        ax.tick_params(labelsize=8)
    for ax in axes.flat[len(QUALITY_DESCRIPTOR_FEATURES) :]:
        ax.set_axis_off()
    fig.suptitle(
        f"Observable world descriptors · n {report.n_worlds} worlds (scale-free statistics of "
        "the model-facing series)"
    )
    plt.show()
    plt.close(fig)


def _fig_quality_text(report: Report) -> None:
    """Acceptance of the realism filter and the stored signal-gate status, as text."""
    evaluated = report.acceptance.get("evaluated")
    rate = report.acceptance.get("rejection_rate")
    if evaluated is None or rate is None:
        print("Acceptance: not recorded.")
    else:
        print(
            f"Acceptance: {1.0 - float(rate):.1%} of {int(evaluated)} evaluated draws passed the "
            f"realism filter ({report.n_worlds} worlds kept)."
        )
    signal = report.corpus["diagnostics"].get("signal")
    if signal is None:
        print("Signal gate: not recorded (no signal diagnostics).")
        return
    ok, lines = check_signal_gate(signal)
    print(f"Signal gate (default thresholds): {'PASS' if ok else 'FAIL'}")
    for line in lines:
        print(f"  {line}")


def plot_quality(report: Report) -> None:
    """#31 checks as plots: prevalence, grid coverage, identifiability, descriptors, text."""
    _fig_quality_prevalence(report)
    _fig_quality_coverage(report)
    _fig_quality_identifiability(report)
    _fig_quality_descriptors(report)
    _fig_quality_text(report)


# ----- exports ----------------------------------------------------------------------
def export_worlds(report: Report, out_dir: Path) -> pd.DataFrame:
    """One model-ready CSV per world plus ``manifest.csv``; returns the manifest (not shown).

    Each CSV holds ``week``, exactly the world's active ``treatment_C*`` and
    ``covariate_Z*`` columns (original slot ids) and ``outcome_Y``, like the
    library's bundle ``dataset.csv``. Every file is read back: its column counts must
    equal the stored active counts and every value must round-trip to the stored
    float32 exactly at CSV precision. The manifest names the truth shard by its
    absolute path and SHA-256, so CSVs stay traceable wherever they are copied.
    """
    corpus = report.corpus
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    truth_path, truth_sha256 = str(report.path.resolve()), _core_sha256(report.path)
    weeks = np.arange(report.T, dtype=np.int64)
    width = len(str(max(report.n_worlds - 1, 0)))
    rows = []
    for world in range(report.n_worlds):
        columns = {"week": weeks}
        n_columns = {}
        for role in _FIG_ROLES:
            slots = np.flatnonzero(active_mask(report, role)[world])
            series = np.asarray(corpus[_FIG_SERIES_KEY[role]][world])
            for slot in slots:
                columns[f"{role}_{_FIG_PREFIX[role]}{slot + 1}"] = series[:, slot]
            n_columns[role] = int(slots.size)
            stored = int(corpus[_FIG_COUNT_KEY[role]][world])
            if n_columns[role] != stored:
                raise AssertionError(
                    f"world {world}: {slots.size} {role} columns, but "
                    f"{_FIG_COUNT_KEY[role]} is {stored}"
                )
        columns["outcome_Y"] = np.asarray(corpus["outcome_raw"][world])
        frame = pd.DataFrame(columns)
        name = f"world-{world:0{width}d}.csv"
        frame.to_csv(out_dir / name, index=False)
        back = pd.read_csv(out_dir / name, float_precision="round_trip")
        if list(back.columns) != list(frame.columns):
            raise AssertionError(f"{name}: columns changed on the CSV round trip")
        if not np.array_equal(back["week"].to_numpy(), weeks):
            raise AssertionError(f"{name}: week column changed on the CSV round trip")
        for column in list(frame.columns)[1:]:
            written = back[column].to_numpy(dtype=np.float64).astype(np.float32)
            if not np.array_equal(written, frame[column].to_numpy()):
                raise AssertionError(f"{name}: {column} differs from the corpus at CSV precision")
        rows.append(
            {
                "dataset": f"{report.path.stem}:{world}",
                "world_id": world,
                "cell_id": int(corpus["cell_id"][world]),
                "n_channels": n_columns["treatment"],
                "n_controls": n_columns["covariate"],
                "n_weeks": report.T,
                "model_data": name,
                "truth_source": truth_path,
                "truth_sha256": truth_sha256,
            }
        )
    manifest = pd.DataFrame(rows)
    if len(manifest) != report.n_worlds or not manifest["model_data"].is_unique:
        raise AssertionError("the manifest must list every world once")
    manifest.to_csv(out_dir / "manifest.csv", index=False)
    print(
        f"Exported {len(manifest)} model-ready CSVs and manifest.csv ({len(manifest)} rows): "
        f"week + {manifest['n_channels'].min()}–{manifest['n_channels'].max()} channel columns "
        f"+ {manifest['n_controls'].min()}–{manifest['n_controls'].max()} control columns + "
        "outcome_Y per world; column counts equal the stored active counts and every value "
        "round-trips exactly at CSV precision."
    )
    return manifest

## The knobs

Everything the cohort depends on is in the next cell: counts, effect sizes, input levels and volatility, noise, texture, trend/seasonal/jump forms, flighting, carryover and saturation. Each range is a uniform law unless its comment says otherwise (Michaelis–Menten κ and the channel level-jump factor are log-uniform, the flighting period is a uniform integer). Some values are fixed by the design — one world per cell, relative noise scales, and no onset or offset gates on channels or controls — and the cell refuses to generate if they change. Other deliberate choices:

- **Counts.** One generation cell per world, so no two worlds share a structure. With 10 worlds every channel count 1–10 and every control count 1–10 appears exactly once, paired so the two counts are nearly uncorrelated (the cell prints their rank correlation); with 100 worlds every pair appears once.
- **Edges.** Every active channel and every active control affects the outcome directly; nothing else is connected, and the latent factor is disconnected.
- **Trend.** Every input carries a trend parameter drawn from −1 to 1; values within ±1/3 read as flat, so the configured law gives equal thirds.
- **Zeros.** Only flighting switches inputs off (on for W of every P weeks). Its ranges were chosen so the number of zero weeks per input is close to even from 0 to 100.
- **Effects.** Contributions are not targeted: each input draws its contribution at its reference level from 0 to 3, the baseline level from 0.05 to 8, and the shares are whatever those draws produce.

In [ ]:
# Every range, law and probability the cohort uses. Edit, re-run, compare.
# Ranges are uniform laws unless a comment says otherwise.
N_WORLDS = 10  # any positive multiple of 10; 100 gives every (channels, controls) pair once
FLAT_BAND = 1 / 3  # |trend change| <= FLAT_BAND reads as "flat" (any number >= 0)

COUNT_RANGES = {"n_treatments_active_range": (1, 10), "n_covariates_active_range": (1, 10)}
PER_INPUT = {  # applied to channels and controls alike
    # Zeros come only from flighting: on for W of every P weeks, at a random phase.
    "flighting_inclusion_prob": 10 / 11,
    "flighting_period_weeks_range": (14, 26),  # P: integer uniform
    "flighting_duty_range": (0.03, 0.99),  # W = clip(floor(duty * P + 0.5), 1, P - 1)
    "onset_inclusion_prob": 0.0,  # fixed by the design (checked below)
    "offset_inclusion_prob": 0.0,  # fixed by the design (checked below)
    "trend_inclusion_prob": 1.0,
    "seasonal_inclusion_prob": 0.5,
    "seasonal_period_weeks_range": (52.0, 52.0),
    "level_jump_inclusion_prob": 0.5,
    "level_jump_count": 1,  # held steps per input that carries a jump
    "pulse_inclusion_prob": 0.5,
    "hf_inclusion_prob": 1.0,
}
KNOBS = {
    "n_treatments": 10,
    "n_covariates": 10,
    "n_latent": 1,
    "n_time_steps": 104,
    **COUNT_RANGES,
    "draws_per_cell": 1,  # fixed by the design (checked below)
    "outcome_std_mode": "relative",  # fixed by the design (checked below)
    # Effect sizes: each input's contribution at its reference level, and the baseline level.
    "treatment_reference_contribution_range": (0.0, 3.0),
    "covariate_reference_contribution_range": (0.0, 3.0),
    "covariate_reference_scale": 1.0,
    "rw_baseline_mean_range": (0.05, 8.0),
    "baseline_floor": 0.0,
    # Input levels, volatility and smoothness.
    "rw_positive_mean_range": (0.1, 8.0),  # channel walk mean, before softplus
    "rw_treatment_std_range": (0.1, 1.2),  # channel walk std, relative to its level
    "rw_covariate_mean_range": (0.1, 2.0),
    "rw_std_sigma": 0.5,  # control walk std ~ HalfNormal(rw_std_sigma)
    "rw_smoothness_alpha": 1.0,  # walk smoothness ~ Beta(alpha, beta)
    "rw_smoothness_beta": 1.0,
    "rw_smoothness_max_weeks": 26,  # smoothness 1 = a 26-week moving average
    # Baseline walk and outcome noise std, relative to the channels' joint effect size.
    "rw_baseline_std_range": (0.0, 0.093),
    "rw_outcome_std_range": (0.01, 0.028),
    # Texture (the make_scm_prior defaults, echoed so every range is visible here).
    "treatment_hf_sigma_range": (0.08, 0.6),
    "treatment_pulse_prob_range": (0.0, 0.25),
    "treatment_pulse_amp_range": (0.4, 2.5),
    "covariate_hf_sigma_range": (0.1, 0.8),
    "covariate_pulse_prob_range": (0.0, 0.25),
    "covariate_pulse_amp_range": (0.5, 3.0),
    # Level forms.
    "treatment_trend_log_change_range": (-1.0, 1.0),
    "covariate_trend_change_range": (-1.0, 1.0),
    "treatment_seasonal_amplitude_range": (0.1, 0.5),
    "covariate_seasonal_amplitude_range": (0.2, 1.0),
    "treatment_level_jump_factor_range": (0.5, 2.0),  # log-uniform
    "covariate_level_jump_size_range": (-1.0, 1.0),
    # Response mechanisms: geometric carryover and Michaelis–Menten saturation.
    "carryover_alpha_range": (0.05, 0.95),
    **{
        f"{role}_{key}": value
        for role in ("treatment", "covariate")
        for key, value in PER_INPUT.items()
    },
}
MM_KAPPA_MULT_RANGE = (0.05, 20.0)  # log-uniform

# The report asserts these design values after loading the shard; fail before generating.
DESIGN_FIXED = {
    "draws_per_cell": (1, "one world per generation cell, so no two worlds share a structure"),
    "outcome_std_mode": ("relative", "the baseline-walk and noise ranges above are relative"),
    **{
        f"{role}_{gate}_inclusion_prob": (0.0, "zeros come only from flighting (exact zero law)")
        for role in ("treatment", "covariate")
        for gate in ("onset", "offset")
    },
}
for key, (value, why) in DESIGN_FIXED.items():
    if KNOBS[key] != value:
        raise ValueError(f"{key} must stay {value!r} in this design: {why}")
if not (math.isfinite(FLAT_BAND) and FLAT_BAND >= 0.0):
    raise ValueError(f"FLAT_BAND must be a finite number >= 0, got {FLAT_BAND!r}")


def count_weights(n_worlds, count_ranges, seed, *, max_tries=2000, target_rho=0.1):
    """Integer count design W: every channel count and every control count equally often.

    Up to side * side worlds no (channels, controls) pair repeats; beyond that W adds
    whole copies of the full grid.
    """
    (t_lo, t_hi), (c_lo, c_hi) = count_ranges.values()
    side = t_hi - t_lo + 1
    if c_hi - c_lo + 1 != side:
        raise ValueError("count_weights needs equally long channel and control count ranges")
    if not isinstance(n_worlds, int) or n_worlds < side or n_worlds % side:
        raise ValueError(f"N_WORLDS must be a positive multiple of {side}, got {n_worlds!r}")
    full, remainder = divmod(n_worlds, side * side)
    shifts = remainder // side

    def design(perm):
        weights = np.full((side, side), full, dtype=np.int64)
        for k in range(shifts):
            weights[np.arange(side), (perm + k) % side] += 1
        return weights

    def rho(weights):
        rows, cols = np.nonzero(weights)
        reps = weights[rows, cols]
        x, y = np.repeat(rows, reps), np.repeat(cols, reps)
        if np.unique(x).size < 2 or np.unique(y).size < 2:
            return float("nan")
        return float(stats.spearmanr(x, y).statistic)

    if not shifts:
        return design(np.arange(side)), float("nan") if side == 1 else 0.0
    if side <= 6:
        candidates = (np.array(p) for p in itertools.permutations(range(side)))
    else:
        rng = np.random.default_rng(seed)
        candidates = (rng.permutation(side) for _ in range(max_tries))
    best = None
    for perm in candidates:
        weights = design(perm)
        value = rho(weights)
        if best is None or abs(value) < abs(best[1]):
            best = (weights, value)
        if abs(value) <= target_rho:
            break
    return best


COUNT_WEIGHTS, COUNT_RHO = count_weights(N_WORLDS, COUNT_RANGES, GENERATION_SEED)
assert int(COUNT_WEIGHTS.sum()) == N_WORLDS

shapes = pg.SCMPrior().saturation_prior_ranges
shapes["michaelis_menten"]["kappa_mult"] = MM_KAPPA_MULT_RANGE
edges = dict.fromkeys(("cy", "dc", "dz", "dy", "zy", "zc", "cc", "zz"), (0, 0))
edges.update(cy=(KNOBS["n_treatments"],) * 2, zy=(KNOBS["n_covariates"],) * 2)
cfg = pg.make_scm_prior(
    **KNOBS,
    n_cells=N_WORLDS,
    seed=GENERATION_SEED,
    edge_budget=edges,
    trajectories="composable",
    active_count_allocation="stratified",
    active_count_weights=COUNT_WEIGHTS.tolist(),
    carryover_family_probs={"none": 0.0, "geometric": 1.0, "weibull": 0.0},
    saturation_family_probs={
        name: float(name == "michaelis_menten") for name in SATURATION_FAMILY_KEYS
    },
    saturation_prior_ranges=shapes,
    mm_scale_prior="log_uniform",
    record_walk_parameters=True,
)
print(
    f"{N_WORLDS} worlds, one per generation cell; channel/control count rank correlation "
    f"{COUNT_RHO:+.2f}; expected generation time ≈ {1.1 * N_WORLDS:.0f} s"
)

In [ ]:
generated = pg.sample_prior_predictive(cfg)
CORPUS_PATH = RUN_DIR / f"variety-{N_WORLDS}-worlds.npz"
pg.save_corpus(generated, CORPUS_PATH)
SIDECAR_PATH = write_sidecar(
    CORPUS_PATH,
    cfg,
    knobs={
        "N_WORLDS": N_WORLDS,
        "FLAT_BAND": FLAT_BAND,
        "GENERATION_SEED": GENERATION_SEED,
        "VISUAL_SEED": VISUAL_SEED,
        "count_weights": COUNT_WEIGHTS.tolist(),
    },
)
corpus = pg.load_corpus(CORPUS_PATH)
assert not pg.DataGenerator.validate_corpus(corpus)
for key, value in generated.items():
    if isinstance(value, np.ndarray):
        np.testing.assert_array_equal(corpus[key], value, err_msg=key)
    elif key == "identifiability":
        for name, array in value.items():
            np.testing.assert_array_equal(corpus[key][name], array, err_msg=name)
assert corpus["diagnostics"] == {k: v for k, v in generated["diagnostics"].items() if k != "timing"}
print("Saved, reloaded and validated", CORPUS_PATH.name, "with", SIDECAR_PATH.name)
print("Output directory:", RUN_DIR.resolve())

## Report from the saved file

From here on, every figure reads only the saved `.npz` and the `.config.json` written next to it. To report on another shard without generating anything, set `PYMC_GENERATOR_REPORT_PATH` to its absolute path and run only the cells tagged `report-setup`, `report-definitions` and `report-run` ([recipe](https://pymc-labs.github.io/pymc-generator/guide/corpus-quality/#report-on-an-existing-shard)); if the variable is still set during a full run, the report says that it reads that shard instead of the new one. Without the config file, or for a shard generated without `record_walk_parameters=True`, the figures still draw what the data show and say which references are unavailable.

**How to read the references.** Black dots are the counts the configured law expects at this N; black whiskers are a family-wise envelope, so a whole panel stays inside it at least 90% of the time when the generator follows its law (more at small N, because exact binomial quantiles are conservative), and bars that leave it are outlined in red. Titles report how many bins fall outside, and the parameter grid prints how many panels chance alone is expected to push outside. The references describe the law before the generator's realism filter, which rejected the share of candidates printed below.

In [ ]:
report_override = os.environ.get("PYMC_GENERATOR_REPORT_PATH")
if not report_override and "CORPUS_PATH" not in globals():
    raise RuntimeError(
        "report-only run: set PYMC_GENERATOR_REPORT_PATH to the absolute path of a saved .npz shard"
    )
REPORT_PATH = Path(report_override) if report_override else CORPUS_PATH
if "CORPUS_PATH" in globals() and REPORT_PATH.resolve() != CORPUS_PATH.resolve():
    print(
        "· PYMC_GENERATOR_REPORT_PATH is set: this report and its section-8 CSVs come from "
        f"{REPORT_PATH}, not the shard saved above"
    )
report = load_report(REPORT_PATH)
for note in report.notes:
    print("·", note)

## 1. How many channels and controls?

In [ ]:
plot_count_grid(report)

## 2. Every parameter that generates the data

One panel per parameter, pooled over worlds (world-level parameters) or over the active inputs that carry the component (per-input parameters). Each panel is drawn on the scale its prior uses — relative volatilities divided by their anchor, log-uniform scales on a log axis, flighting as the fraction of weeks on. Derived quantities are labelled and carry no reference.

In [ ]:
plot_parameter_grid(report)

## 3. Trend: flat, up or down

Two figures, channels on the left and controls on the right. The first shows the trend parameter the generator drew: by value (top), which the configured law spreads evenly over −1 to 1 (channels on a log-level scale, controls in their own units), and split into down, flat and up (bottom), which the law makes equal thirds. The second shows the direction each series actually takes over its on-weeks (top) and each input's parameter against its realized change (bottom). Random-walk drift, seasonality and level jumps move the realized direction too, so the two need not agree.

In [ ]:
plot_trend(report)

## 4. How many weeks are exactly zero?

Each active input counts its exactly-zero weeks. The categories run from no zeros to 100; under the configured flighting law every category is about equally likely, and 101–104 cannot happen because every flight keeps at least one on-week per period.

In [ ]:
plot_zero_weeks(report)

## 5. Where does the outcome come from?

For each world: the channels' share of the summed outcome, the controls' share, and the baseline's share (intercept and observation noise). The three add up to 100%, so they cannot all be spread evenly at once: if every split of the 100% were equally likely, each single share would fall off linearly from 0% to 100%. That reference is drawn as a guide — it is an ideal, not the generator's law. Controls have non-negative effects, but their series are signed, so a world's control share can be slightly negative.

In [ ]:
plot_shares(report)

## 6. Example worlds

The worlds with the lowest and highest channel, control and baseline shares (a world that is extreme twice is shown once), plus one seeded pick. Top: the inputs; bottom: the outcome split into its exact parts.

In [ ]:
for world_id, labels in pick_example_worlds(report):
    plot_world(report, world_id, labels)

## 7. Quality checks

The corpus report of #31, as plots: how often each trajectory component is carried, count-grid coverage, an empirical identifiability screen (a world is flagged when its observed inputs are rank-deficient or any variance-inflation factor is at least 10, on week-to-week differences; the cohort passes when at most 25% of worlds are flagged), descriptor distributions, and the generator's own acceptance and signal checks. The identifiability screen is a finite-sample redundancy check of the raw inputs, not proof of causal identifiability.

In [ ]:
plot_quality(report)

## 8. Keep the datasets

One model-ready CSV per world — only the active channel and control columns plus the outcome — and a manifest, written to `datasets/<shard name>/` in this run's output directory (printed after generation; set `PYMC_GENERATOR_SHOWCASE_DIR` to choose its parent). The manifest names the truth shard by its absolute path and SHA-256. Report-only runs export the shard they read the same way. Truth stays in the `.npz`.

In [ ]:
manifest = export_worlds(report, RUN_DIR / "datasets" / REPORT_PATH.stem)

## What to change

- More worlds: set `N_WORLDS = 100` (every channel/control pair once) or any multiple of 10.
- A different share mix: the baseline range and the two reference-contribution ranges set where the composition points fall.
- Realized trends closer to the parameter: narrower walk volatilities (`rw_treatment_std_range`, `rw_std_sigma`).
- More or fewer zeros: the flighting inclusion probability, period and duty ranges.